In [1]:
! pip install lxml
! pip install pandas
! pip install bs4


^C
Traceback (most recent call last):
  File "/opt/homebrew/anaconda3/bin/pip", line 11, in <module>
    sys.exit(main())
             ^^^^^^
  File "/opt/homebrew/anaconda3/lib/python3.12/site-packages/pip/_internal/cli/main.py", line 79, in main
    return command.main(cmd_args)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/anaconda3/lib/python3.12/site-packages/pip/_internal/cli/base_command.py", line 101, in main
    return self._main(args)
           ^^^^^^^^^^^^^^^^
  File "/opt/homebrew/anaconda3/lib/python3.12/site-packages/pip/_internal/cli/base_command.py", line 236, in _main
    self.handle_pip_version_check(options)
  File "/opt/homebrew/anaconda3/lib/python3.12/site-packages/pip/_internal/cli/req_command.py", line 177, in handle_pip_version_check
    session = self._build_session(
              ^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/anaconda3/lib/python3.12/site-packages/pip/_internal/cli/req_command.py", line 122, in _build_session
    session = PipSession(


In [ ]:
import os

# Path to the original XML file
file_path = './data/a.xml'
temp_file_path = './data/a_modified.xml'

# DTD content to insert after the XML declaration
dtd_content = '''<!DOCTYPE PubmedArticleSet [
<!ENTITY quote "&#34;">
<!ENTITY apos "&#39;">
<!ENTITY amp "&#38;">
<!ENTITY lt "&#60;">
<!ENTITY gt "&#62;">
]>'''

# Open the original file for reading
with open(file_path, 'r', encoding='utf-8') as original_file:
    # Read the first line
    first_line = original_file.readline().strip()

    # Check if the first line contains the XML declaration
    if first_line.startswith('<?xml'):
        # Create a temporary file for writing
        with open(temp_file_path, 'w', encoding='utf-8') as temp_file:
            # Write the XML declaration and DTD to the temporary file
            temp_file.write(f"{first_line}\n{dtd_content}\n")

            # Copy the rest of the original file to the temporary file
            chunk_size = 1024 * 1024  # 1MB chunks
            while True:
                chunk = original_file.read(chunk_size)
                if not chunk:
                    break
                temp_file.write(chunk)

        # Replace the original file with the modified temporary file
        os.replace(temp_file_path, file_path)
    else:
        # If the first line is not an XML declaration, handle it accordingly
        print("The file does not start with an XML declaration. Unable to add DTD.")

print("Successfully replaced the first line with XML declaration and DTD in the same file.")


Successfully replaced the first line with XML declaration and DTD in the same file.


In [ ]:
from lxml import etree
import pandas as pd
import html

# Function to decode HTML entities in text


def decode_html_entities(text):
    return html.unescape(text) if text else ''

# Function to parse each PubmedArticle element


def parse_pubmed_article(element):
    data = {}

    # Extract data from MedlineCitation
    medline_citation = element.find('MedlineCitation')
    if medline_citation is not None:
        data['PMID'] = medline_citation.findtext('PMID')

        data['MedlineCitationOwner'] = medline_citation.get('Owner', '')
        data['MedlineCitationStatus'] = medline_citation.get('Status', '')
        data['MedlineCitationVersionID'] = medline_citation.get(
            'VersionID', '')
        data['MedlineCitationVersionDate'] = medline_citation.get(
            'VersionDate', '')
        data['MedlineCitationIndexingMethod'] = medline_citation.get(
            'IndexingMethod', '')

        date_completed = medline_citation.find('DateCompleted')
        if date_completed is not None:
            data['DateCompleted'] = {
                'Year': date_completed.findtext('Year'),
                'Month': date_completed.findtext('Month'),
                'Day': date_completed.findtext('Day')
            }

        date_revised = medline_citation.find('DateRevised')
        if date_revised is not None:
            data['DateRevised'] = {
                'Year': date_revised.findtext('Year'),
                'Month': date_revised.findtext('Month'),
                'Day': date_revised.findtext('Day')
            }

        article = medline_citation.find('Article')
        if article is not None:
            data['ArticlePubModel'] = article.get('PubModel', '')

            journal = article.find('Journal')
            if journal is not None:
                data['JournalISSN'] = journal.findtext('ISSN')
                data['JournalTitle'] = journal.findtext('Title')
                data['JournalISOAbbreviation'] = journal.findtext(
                    'ISOAbbreviation')

            data['ArticleTitle'] = decode_html_entities(
                article.findtext('ArticleTitle'))
            data['Language'] = article.findtext('Language')

            pagination = article.find('Pagination')
            if pagination is not None:
                data['StartPage'] = pagination.findtext('StartPage')
                data['EndPage'] = pagination.findtext('EndPage')
                data['MedlinePgn'] = pagination.findtext('MedlinePgn')

            abstract = article.find('Abstract')
            if abstract is not None:
                abstract_texts = abstract.findall('AbstractText')
                data['Abstract'] = [{'Label': text.get('Label', ''),
                                     'NlmCategory': text.get('NlmCategory', ''),
                                     'Text': decode_html_entities(text.text)}
                                    for text in abstract_texts]

            authors = []
            for author_elem in article.findall('AuthorList/Author'):
                author = {
                    'LastName': author_elem.findtext('LastName'),
                    'ForeName': author_elem.findtext('ForeName'),
                    'Initials': author_elem.findtext('Initials'),
                    'Affiliation': []
                }
                # Extract affiliations for each author
                affiliations = author_elem.findall(
                    'AffiliationInfo/Affiliation')
                for affiliation_elem in affiliations:
                    author['Affiliation'].append(
                        decode_html_entities(affiliation_elem.text))
                authors.append(author)
            data['Authors'] = authors

            data['DataBankList'] = []
            databanks = article.findall('DataBankList/DataBank')
            for databank in databanks:
                data_bank = {
                    'DataBankName': databank.findtext('DataBankName'),
                    'AccessionNumberList': [accession.text for accession in databank.findall('AccessionNumberList/AccessionNumber')]
                }
                data['DataBankList'].append(data_bank)

            data['GrantList'] = []
            grants = article.findall('GrantList/Grant')
            for grant in grants:
                grant_data = {
                    'GrantID': grant.findtext('GrantID'),
                    'Acronym': grant.findtext('Acronym'),
                    'Agency': grant.findtext('Agency'),
                    'Country': grant.findtext('Country')
                }
                data['GrantList'].append(grant_data)

            publication_types = article.findall(
                'PublicationTypeList/PublicationType')
            data['PublicationTypeList'] = [
                ptype.text for ptype in publication_types]

            vernacular_title = article.find('VernacularTitle')
            if vernacular_title is not None:
                data['VernacularTitle'] = decode_html_entities(
                    vernacular_title.text)

            article_dates = article.findall('ArticleDate')
            data['ArticleDates'] = [{
                'Year': date.findtext('Year'),
                'Month': date.findtext('Month'),
                'Day': date.findtext('Day'),
                'DateType': date.get('DateType', '')
            } for date in article_dates]

        # Extract data from PubmedData
        pubmed_data = element.find('PubmedData')
        if pubmed_data is not None:
            data['PublicationStatus'] = pubmed_data.findtext(
                'PublicationStatus')

            article_ids = pubmed_data.findall('ArticleIdList/ArticleId')
            data['ArticleIds'] = [{
                'IdType': article_id.get('IdType', ''),
                'Value': article_id.text
            } for article_id in article_ids]

            history = pubmed_data.findall('History/PubMedPubDate')
            data['History'] = [{
                'PubStatus': date.get('PubStatus', ''),
                'Year': date.findtext('Year'),
                'Month': date.findtext('Month'),
                'Day': date.findtext('Day'),
                'Hour': date.findtext('Hour'),
                'Minute': date.findtext('Minute'),
                'Second': date.findtext('Second')
            } for date in history]

            objects = pubmed_data.findall('ObjectList/Object')
            data['Objects'] = [{
                'ObjectType': obj.get('Type', ''),
                'Params': {param.get('Name'): param.text for param in obj.findall('Param')}
            } for obj in objects]

    return data


# Initialize a list to hold parsed data
parsed_data = []

# Parse the XML file using iterparse for large files
parser = etree.iterparse('./data/pubmed_ml_ai_results.xml', events=('end',),
                         tag='PubmedArticle', encoding='utf-8', recover=True)

for event, elem in parser:
    article_data = parse_pubmed_article(elem)
    parsed_data.append(article_data)
    elem.clear()  # Clear the element to save memory

# Create a DataFrame from parsed data
df = pd.DataFrame(parsed_data)

# Print the DataFrame
print(df)


            PMID MedlineCitationOwner MedlineCitationStatus  \
0       17379690                  NLM    PubMed-not-MEDLINE   
1       20585802                  NLM               MEDLINE   
2       26254699                  NLM               MEDLINE   
3       26394433                  NLM               MEDLINE   
4       26409750                  NLM               MEDLINE   
...          ...                  ...                   ...   
203226  39017971                  NLM             Publisher   
203227  39018014                  NLM             Publisher   
203228  39018022                  NLM             Publisher   
203229  39018040                  NLM             Publisher   
203230  39018100                  NLM             Publisher   

       MedlineCitationVersionID MedlineCitationVersionDate  \
0                                                            
1                                                            
2                                                        

In [ ]:
parsed_data[2]


{'PMID': '26254699',
 'MedlineCitationOwner': 'NLM',
 'MedlineCitationStatus': 'MEDLINE',
 'MedlineCitationVersionID': '',
 'MedlineCitationVersionDate': '',
 'MedlineCitationIndexingMethod': '',
 'DateCompleted': {'Year': '2019', 'Month': '11', 'Day': '01'},
 'DateRevised': {'Year': '2019', 'Month': '11', 'Day': '01'},
 'ArticlePubModel': 'Print-Electronic',
 'JournalISSN': '1873-2860',
 'JournalTitle': 'Artificial intelligence in medicine',
 'JournalISOAbbreviation': 'Artif Intell Med',
 'ArticleTitle': 'Origins of the Arden Syntax.',
 'Language': 'eng',
 'StartPage': '7',
 'EndPage': '9',
 'MedlinePgn': '7-9',
 'Abstract': [{'Label': '',
   'NlmCategory': '',
   'Text': "The Arden Syntax originated in the 1980's, when several knowledge-based systems began to show promise, but researchers recognized the burden of recreating these systems at every institution. Derived initially from Health Evaluation through Logical Processing (HELP) and the Regenstrief Medical Record System (RMRS), t

In [ ]:
# Example PubMed IDs to filter
pubmed_ids_to_filter = ['37045921', '37016152',
                        '37720336', '38083058', '34891618', '32750906']

# Filter the DataFrame based on PubMed IDs
filtered_df = df[df['PMID'].isin(pubmed_ids_to_filter)]

# Print the filtered DataFrame
filtered_df


,PMID,MedlineCitationOwner,MedlineCitationStatus,MedlineCitationVersionID,MedlineCitationVersionDate,MedlineCitationIndexingMethod,DateRevised,ArticlePubModel,JournalISSN,JournalTitle,...,DataBankList,GrantList,PublicationTypeList,ArticleDates,PublicationStatus,ArticleIds,History,Objects,DateCompleted,VernacularTitle
42216,32750906,NLM,MEDLINE,,,Automated,"{'Year': '2021', 'Month': '09', 'Day': '24'}",Print-Electronic,2168-2208,IEEE journal of biomedical and health informatics,...,[],[],"[Journal Article, Research Support, Non-U.S. G...","[{'Year': '2021', 'Month': '03', 'Day': '05', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '32750906'}, {'...","[{'PubStatus': 'pubmed', 'Year': '2020', 'Mont...",[],"{'Year': '2021', 'Month': '09', 'Day': '24'}",NaN
85429,34891618,NLM,MEDLINE,,,Automated,"{'Year': '2021', 'Month': '12', 'Day': '29'}",Print,2694-0604,Annual International Conference of the IEEE En...,...,[],[],"[Journal Article, Research Support, Non-U.S. G...",[],ppublish,"[{'IdType': 'pubmed', 'Value': '34891618'}, {'...","[{'PubStatus': 'entrez', 'Year': '2021', 'Mont...",[],"{'Year': '2021', 'Month': '12', 'Day': '29'}",NaN
140437,37016152,NLM,PubMed-not-MEDLINE,,,Automated,"{'Year': '2023', 'Month': '04', 'Day': '11'}",Electronic,2398-6352,NPJ digital medicine,...,[],[],[Journal Article],"[{'Year': '2023', 'Month': '04', 'Day': '04', ...",epublish,"[{'IdType': 'pubmed', 'Value': '37016152'}, {'...","[{'PubStatus': 'received', 'Year': '2022', 'Mo...",[],NaN,NaN
160951,37720336,NLM,PubMed-not-MEDLINE,,,Automated,"{'Year': '2023', 'Month': '09', 'Day': '20'}",Electronic-eCollection,2666-3899,"Patterns (New York, N.Y.)",...,[],[],[Journal Article],"[{'Year': '2023', 'Month': '08', 'Day': '03', ...",epublish,"[{'IdType': 'pubmed', 'Value': '37720336'}, {'...","[{'PubStatus': 'received', 'Year': '2022', 'Mo...",[],NaN,NaN
172145,38083058,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '01', 'Day': '26'}",Print,2694-0604,Annual International Conference of the IEEE En...,...,[],[],"[Journal Article, Research Support, Non-U.S. G...",[],ppublish,"[{'IdType': 'pubmed', 'Value': '38083058'}, {'...","[{'PubStatus': 'medline', 'Year': '2023', 'Mon...",[],"{'Year': '2023', 'Month': '12', 'Day': '16'}",NaN


In [ ]:
df


,PMID,MedlineCitationOwner,MedlineCitationStatus,MedlineCitationVersionID,MedlineCitationVersionDate,MedlineCitationIndexingMethod,DateRevised,ArticlePubModel,JournalISSN,JournalTitle,...,DataBankList,GrantList,PublicationTypeList,ArticleDates,PublicationStatus,ArticleIds,History,Objects,DateCompleted,VernacularTitle
0,17379690,NLM,PubMed-not-MEDLINE,,,,"{'Year': '2020', 'Month': '05', 'Day': '19'}",Print,1367-4811,"Bioinformatics (Oxford, England)",...,[],[],"[Journal Article, Retraction of Publication]",[],ppublish,"[{'IdType': 'pubmed', 'Value': '17379690'}, {'...","[{'PubStatus': 'received', 'Year': '2007', 'Mo...",[],NaN,NaN
1,20585802,NLM,MEDLINE,,,Automated,"{'Year': '2021', 'Month': '10', 'Day': '20'}",Print-Electronic,1435-2451,Langenbeck's archives of surgery,...,[],[],"[Letter, Comment]","[{'Year': '2010', 'Month': '06', 'Day': '29', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '20585802'}, {'...","[{'PubStatus': 'received', 'Year': '2010', 'Mo...",[],"{'Year': '2019', 'Month': '10', 'Day': '21'}",NaN
2,26254699,NLM,MEDLINE,,,,"{'Year': '2019', 'Month': '11', 'Day': '01'}",Print-Electronic,1873-2860,Artificial intelligence in medicine,...,[],[],"[Historical Article, Journal Article]","[{'Year': '2015', 'Month': '07', 'Day': '02', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '26254699'}, {'...","[{'PubStatus': 'received', 'Year': '2014', 'Mo...",[],"{'Year': '2019', 'Month': '11', 'Day': '01'}",NaN
3,26394433,NLM,MEDLINE,,,,"{'Year': '2019', 'Month': '05', 'Day': '20'}",Print-Electronic,1557-9964,IEEE/ACM transactions on computational biology...,...,[],[],"[Journal Article, Research Support, Non-U.S. G...","[{'Year': '2015', 'Month': '09', 'Day': '18', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '26394433'}, {'...","[{'PubStatus': 'pubmed', 'Year': '2015', 'Mont...",[],"{'Year': '2019', 'Month': '05', 'Day': '20'}",NaN
4,26409750,NLM,MEDLINE,,,,"{'Year': '2019', 'Month': '11', 'Day': '01'}",Print-Electronic,1873-2860,Artificial intelligence in medicine,...,[],[],[Journal Article],"[{'Year': '2015', 'Month': '09', 'Day': '12', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '26409750'}, {'...","[{'PubStatus': 'received', 'Year': '2015', 'Mo...",[],"{'Year': '2019', 'Month': '11', 'Day': '01'}",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203226,39017971,NLM,Publisher,,,Automated,"{'Year': '2024', 'Month': '07', 'Day': '17'}",Print-Electronic,1534-4681,Annals of surgical oncology,...,[],[],[Letter],"[{'Year': '2024', 'Month': '07', 'Day': '17', ...",aheadofprint,"[{'IdType': 'pubmed', 'Value': '39017971'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],NaN,NaN
203227,39018014,NLM,Publisher,,,Automated,"{'Year': '2024', 'Month': '07', 'Day': '17'}",Print-Electronic,1460-2156,Brain : a journal of neurology,...,[],[],[Journal Article],"[{'Year': '2024', 'Month': '07', 'Day': '17', ...",aheadofprint,"[{'IdType': 'pubmed', 'Value': '39018014'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],NaN,NaN
203228,39018022,NLM,Publisher,,,Automated,"{'Year': '2024', 'Month': '07', 'Day': '17'}",Print-Electronic,1464-3677,International journal for quality in health ca...,...,[],[],[Journal Article],"[{'Year': '2024', 'Month': '07', 'Day': '17', ...",aheadofprint,"[{'IdType': 'pubmed', 'Value': '39018022'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],NaN,NaN
203229,39018040,NLM,Publisher,,,Automated,"{'Year': '2024', 'Month': '07', 'Day': '17'}",Print-Electronic,2380-6591,JAMA cardiology,...,[],[],[Journal Article],"[{'Year': '2024', 'Month': '07', 'Day': '17', ...",aheadofprint,"[{'IdType': 'pubmed', 'Value': '39018040'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],NaN,NaN


In [ ]:
# Initialize a list to hold parsed data
parsed_data_2025 = []

# Parse the XML file using iterparse for large files
parser_2025 = etree.iterparse('./data/pubmed_ml_ai_uid_add_2025.text', events=('end',),
                              tag='PubmedArticle', encoding='utf-8', recover=True)

for event, elem in parser_2025:
    article_data = parse_pubmed_article(elem)
    parsed_data_2025.append(article_data)
    elem.clear()  # Clear the element to save memory

# Create a DataFrame from parsed data
df_2025 = pd.DataFrame(parsed_data_2025)

df_2025


,PMID,MedlineCitationOwner,MedlineCitationStatus,MedlineCitationVersionID,MedlineCitationVersionDate,MedlineCitationIndexingMethod,DateCompleted,DateRevised,ArticlePubModel,JournalISSN,...,DataBankList,GrantList,PublicationTypeList,ArticleDates,PublicationStatus,ArticleIds,History,Objects,Abstract,VernacularTitle
0,33874846,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '07', 'Day': '19'}","{'Year': '2024', 'Month': '07', 'Day': '19'}",Print-Electronic,1752-6116,...,[],[],[Editorial],"[{'Year': '2021', 'Month': '04', 'Day': '20', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '33874846'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],NaN,NaN
1,34137113,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '05', 'Day': '09'}","{'Year': '2024', 'Month': '08', 'Day': '26'}",Print-Electronic,1522-2586,...,[],"[{'GrantID': 'DPI2017-86696-R', 'Acronym': Non...","[Journal Article, Research Support, Non-U.S. G...","[{'Year': '2021', 'Month': '06', 'Day': '16', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '34137113'}, {'...","[{'PubStatus': 'revised', 'Year': '2021', 'Mon...",[],"[{'Label': 'BACKGROUND', 'NlmCategory': '', 'T...",NaN
2,34779775,NLM,MEDLINE,,,Curated,"{'Year': '2024', 'Month': '10', 'Day': '01'}","{'Year': '2024', 'Month': '11', 'Day': '05'}",Print-Electronic,1018-9068,...,[],[],"[Journal Article, Observational Study]","[{'Year': '2021', 'Month': '04', 'Day': '15', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '34779775'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': 'OBJECTIVE', 'NlmCategory': 'OBJECT...",NaN
3,34792285,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '05', 'Day': '17'}","{'Year': '2024', 'Month': '05', 'Day': '17'}",Print-Electronic,1471-1842,...,[],"[{'GrantID': None, 'Acronym': None, 'Agency': ...",[Journal Article],"[{'Year': '2021', 'Month': '11', 'Day': '18', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '34792285'}, {'...","[{'PubStatus': 'revised', 'Year': '2021', 'Mon...",[],"[{'Label': 'BACKGROUND', 'NlmCategory': 'BACKG...",NaN
4,34847753,NLM,In-Process,,,,NaN,"{'Year': '2025', 'Month': '03', 'Day': '15'}",Print-Electronic,2041-3033,...,[],[],"[Journal Article, Retracted Publication]","[{'Year': '2021', 'Month': '12', 'Day': '01', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '34847753'}, {'...","[{'PubStatus': 'pubmed', 'Year': '2021', 'Mont...",[],NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53796,40455017,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '06', 'Day': '02'}","{'Year': '2025', 'Month': '06', 'Day': '02'}",Print,0023-1207,...,[],[],"[English Abstract, Journal Article]",[],ppublish,"[{'IdType': 'pubmed', 'Value': '40455017'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': 'OBJECTIVE', 'NlmCategory': 'OBJECT...",Prognosticheskaya model' i kal'kulyator otsenk...
53797,40455151,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '06', 'Day': '02'}","{'Year': '2025', 'Month': '06', 'Day': '02'}",Print,1940-6029,...,[],[],[Journal Article],[],ppublish,"[{'IdType': 'pubmed', 'Value': '40455151'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Vir...",NaN
53798,40455152,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '06', 'Day': '02'}","{'Year': '2025', 'Month': '06', 'Day': '02'}",Print,1940-6029,...,[],[],[Journal Article],[],ppublish,"[{'IdType': 'pubmed', 'Value': '40455152'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Und...",NaN
53799,40455154,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '06', 'Day': '02'}","{'Year': '2025', 'Month': '06', 'Day': '02'}",Print,1940-6029,...,[],[],[Journal Article],[],ppublish,"[{'IdType': 'pubmed', 'Value': '40455154'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'The...",NaN


In [ ]:
filtered_df_2025 = df_2025[~df_2025['PMID'].isin(df['PMID'])]
filtered_df_2025


,PMID,MedlineCitationOwner,MedlineCitationStatus,MedlineCitationVersionID,MedlineCitationVersionDate,MedlineCitationIndexingMethod,DateCompleted,DateRevised,ArticlePubModel,JournalISSN,...,DataBankList,GrantList,PublicationTypeList,ArticleDates,PublicationStatus,ArticleIds,History,Objects,Abstract,VernacularTitle
7,35108088,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '10', 'Day': '17'}","{'Year': '2025', 'Month': '04', 'Day': '30'}",Print-Electronic,2167-647X,...,[],[],"[Journal Article, Research Support, Non-U.S. G...","[{'Year': '2022', 'Month': '02', 'Day': '02', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '35108088'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Tra...",NaN
8,35143339,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '12', 'Day': '16'}","{'Year': '2025', 'Month': '04', 'Day': '30'}",Print-Electronic,2167-647X,...,[],[],[Journal Article],"[{'Year': '2022', 'Month': '02', 'Day': '10', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '35143339'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Dee...",NaN
11,35271383,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '12', 'Day': '16'}","{'Year': '2025', 'Month': '04', 'Day': '30'}",Print-Electronic,2167-647X,...,[],[],"[Journal Article, Research Support, Non-U.S. G...","[{'Year': '2022', 'Month': '03', 'Day': '10', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '35271383'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Net...",NaN
15,35385378,NLM,MEDLINE,,,Curated,"{'Year': '2024', 'Month': '10', 'Day': '23'}","{'Year': '2024', 'Month': '11', 'Day': '14'}",Print-Electronic,1949-3614,...,[],[],"[Journal Article, Controlled Clinical Trial]","[{'Year': '2022', 'Month': '05', 'Day': '05', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '35385378'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'The...",NaN
16,35486562,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '08', 'Day': '08'}","{'Year': '2025', 'Month': '01', 'Day': '07'}",Print-Electronic,1557-9964,...,[],"[{'GrantID': 'U01 AG024904', 'Acronym': 'AG', ...","[Journal Article, Research Support, U.S. Gov't...","[{'Year': '2024', 'Month': '08', 'Day': '08', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '35486562'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Alz...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53796,40455017,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '06', 'Day': '02'}","{'Year': '2025', 'Month': '06', 'Day': '02'}",Print,0023-1207,...,[],[],"[English Abstract, Journal Article]",[],ppublish,"[{'IdType': 'pubmed', 'Value': '40455017'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': 'OBJECTIVE', 'NlmCategory': 'OBJECT...",Prognosticheskaya model' i kal'kulyator otsenk...
53797,40455151,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '06', 'Day': '02'}","{'Year': '2025', 'Month': '06', 'Day': '02'}",Print,1940-6029,...,[],[],[Journal Article],[],ppublish,"[{'IdType': 'pubmed', 'Value': '40455151'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Vir...",NaN
53798,40455152,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '06', 'Day': '02'}","{'Year': '2025', 'Month': '06', 'Day': '02'}",Print,1940-6029,...,[],[],[Journal Article],[],ppublish,"[{'IdType': 'pubmed', 'Value': '40455152'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Und...",NaN
53799,40455154,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '06', 'Day': '02'}","{'Year': '2025', 'Month': '06', 'Day': '02'}",Print,1940-6029,...,[],[],[Journal Article],[],ppublish,"[{'IdType': 'pubmed', 'Value': '40455154'}, {'...","[{'PubStatus

In [ ]:
filtered_df_2025.to_csv('data_2025.csv', index=False)


In [ ]:
# Extract authors' affiliations
affiliations = []
for idx, row in df.iterrows():
    for author in row['Authors']:
        for affiliation in author['Affiliation']:
            affiliations.append({
                'PMID': row['PMID'],
                'LastName': author['LastName'],
                'ForeName': author['ForeName'],
                'Initials': author['Initials'],
                'Affiliation': affiliation
            })

# Create a DataFrame for affiliations
affiliations_df = pd.DataFrame(affiliations)


In [ ]:
affiliations_df


,PMID,LastName,ForeName,Initials,Affiliation
0,35108088,Dai,Fei,F,School of Big Data and Intelligent Engineering...
1,35108088,Cao,Pengfei,P,School of Big Data and Intelligent Engineering...
2,35108088,Huang,Penggui,P,School of Big Data and Intelligent Engineering...
3,35108088,Mo,Qi,Q,"School of Software, Yunnan University, Kunming..."
4,35108088,Huang,Bi,B,School of Big Data and Intelligent Engineering...
...,...,...,...,...,...
361934,40455159,Sharma,Amita,A,"Bioclues.org, Pune, India."
361935,40455159,Valadi,Jayaraman K,JK,"Department of Computer Science, FLAME Universi..."
361936,40455159,Valadi,Jayaraman K,JK,"Bioclues.org, Pune, India."
361937,40455159,Suravajhala,Prashanth,P,"Bioclues.org, Pune, India."


In [ ]:
unique_affiliations_df = affiliations_df.drop_duplicates(subset=[
                                                         'Affiliation'])


In [ ]:
unique_affiliations_df


,PMID,LastName,ForeName,Initials,Affiliation
0,35108088,Dai,Fei,F,School of Big Data and Intelligent Engineering...
3,35108088,Mo,Qi,Q,"School of Software, Yunnan University, Kunming..."
5,35143339,Gurusamy,Rajalakshmi,R,"Department of Information Technology, Sethu In..."
7,35271383,Mara,Alexandru Cristian,AC,Department of Electronics and Information Syst...
10,35385378,Abidin,Nihan,N,Clinic of Physical Medicine and Rehabilitation...
...,...,...,...,...,...
361930,40455154,Acharya,Vishal,V,Academy of Scientific and Innovative Research ...
361931,40455159,Bhargava,Harshita,H,"Department of Computer Science and IT, IIS (De..."
361932,40455159,Bhargava,Harshita,H,"Bioclues.org, Pune, India."
361935,40455159,Valadi,Jayaraman K,JK,"Department of Computer Science, FLAME Universi..."


In [ ]:
! pip install langchain
! pip install langchain-community


  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 15.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.1 MB/s eta 0:00:00
Using cached typing_inspect-0.9.0-py3-none-any.whl (8.8 kB)


In [ ]:
unique_affiliations_df.iloc[1]["Affiliation"]


'Department of biostatistics, Faculty of Medical Sciences, Tarbiat Modares university, Tehran, Iran.'

In [ ]:
from langchain_community.llms import Ollama

llm = Ollama(model="llama3")

# Define the prompt text
prompt_text = """
Given this affiliation: 

- {affiliation}

PLEASE GIVE ME ONLY THE NAME OF THE CORRESPONDING COUNTRY. PLEASE REFRAIN FROM USING ABBREVIATIONS FOR COUNTRY NAMES. If you are unable to identify the country, please return "NONE".
"""

# Initialize a list to store the results
countries = []

# Ensure we only process the first 100 rows
num_rows_to_process = min(len(unique_affiliations_df), 20000)

# Iterate over the first 100 rows
for index in range(num_rows_to_process):
    # Get the affiliation item from the current row
    item = unique_affiliations_df.iloc[index]["Affiliation"]
    # Format the prompt with the current affiliation
    prompt_with_item = prompt_text.format(affiliation=item)
    # Invoke the LLM
    country_name = llm.invoke(prompt_with_item, max_tokens=100)
    # Append the country name to the results list
    countries.append({"Affiliation": item, "Country": country_name})

# Print the results
for i, country in enumerate(countries):
    print(f"Row {i+1}: {country}")


Row 1: {'Affiliation': 'Department of biophysics, Faculty of Basic Sciences, Tarbiat Modares University, Tehran, Iran.', 'Country': 'Iran'}
Row 2: {'Affiliation': 'Department of biostatistics, Faculty of Medical Sciences, Tarbiat Modares university, Tehran, Iran.', 'Country': 'Iran'}
Row 3: {'Affiliation': ', Bangkhae, Bangkok, 10160, Thailand. wviroj@yahoo.com.', 'Country': 'Thailand'}
Row 4: {'Affiliation': 'Department of Biomedical Informatics, Columbia University, 622 W 168th Street, PH20, New York, NY 10032, USA; Medical Informatics Services, NewYork-Presbyterian Hospital, 622 W 168th Street, PH20, New York, NY 10032, USA. Electronic address: hripcsak@columbia.edu.', 'Country': 'United States'}
Row 5: {'Affiliation': 'Department of Medical Informatics, Linköping University, 58183 Linköping, Sweden.', 'Country': 'Sweden'}
Row 6: {'Affiliation': 'Department of Biomedical Informatics, Columbia University, 622 W 168th Street, PH20, New York, NY 10032, USA; Department of Biomedical Inf

In [ ]:
df = pd.DataFrame(countries)
df.to_csv('./data/countries_affiliation_20k.csv', index=False)


In [ ]:
unique_affiliations_df


,PMID,LastName,ForeName,Initials,Affiliation
0,17379690,Mehdi,Poursheikhali Asgary,PA,"Department of biophysics, Faculty of Basic Sci..."
2,17379690,Anoshirvan,Kazemnejad,K,"Department of biostatistics, Faculty of Medica..."
4,20585802,Wiwanitkit,Viroj,V,", Bangkhae, Bangkok, 10160, Thailand. wviroj@y..."
5,26254699,Hripcsak,George,G,"Department of Biomedical Informatics, Columbia..."
6,26254699,Wigertz,Ove B,OB,"Department of Medical Informatics, Linköping U..."
...,...,...,...,...,...
1587872,39018040,Kroon,Jeffrey,J,Laboratory of Angiogenesis and Vascular Metabo...
1587873,39018040,Planken,R Nils,RN,"Department of Radiology and Nuclear Medicine, ..."
1587882,39018100,Anmella,Gerard,G,"Bipolar and Depressive Disorders Unit, Departm..."
1587885,39018100,Anmella,Gerard,G,"Departament de Medicina, Facultat de Medicina ..."


In [ ]:
canada_affiliations_df = unique_affiliations_df[unique_affiliations_df['Affiliation'].str.contains(
    'Canada', case=False, na=False)]


In [ ]:
len(unique_affiliations_df)


681447

In [ ]:
canada_affiliations_df


,PMID,LastName,ForeName,Initials,Affiliation
532,28350526,Lindsay,Sally,S,a Department of Occupational Science and Occup...
533,28350526,Lam,Ashley,A,"b Health Sciences, McMaster University , Hamil..."
692,28444633,Shahnazian,Danesh,D,"Department of Psychology, University of Victor..."
693,28444633,Holroyd,Clay B,CB,"Department of Psychology, University of Victor..."
866,28550657,Hollis,Geoff,G,"Department of Psychology, University of Albert..."
...,...,...,...,...,...
1587140,39013867,Ding,Jun,J,"School of Computer Science, McGill University,..."
1587141,39013867,Ding,Jun,J,"Mila-Quebec AI Institute, 6666 Rue Saint-Urbai..."
1587419,39015068,Wolter,Nikolaus E,NE,Department of Otolaryngology- Head and Neck Su...
1587460,39015268,Choudhary,Ruhi,R,Department of Chemical Engineering and Applied...


In [ ]:
! pip install pycountry
! pip install pandas geopy geonamescache


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.4/125.4 kB 2.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 5.7 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.0 MB/s eta 0:00:00


In [ ]:
gc.get_countries()


NameError: name 'gc' is not defined

In [ ]:
import re
import geonamescache

gc = geonamescache.GeonamesCache()

# Get country names from geonamescache
countries = gc.get_countries()
country_names = {country['name'] for country in countries.values()}

# Get US states from geonamescache
us_states = gc.get_us_states()
us_state_names = {state['name'] for state in us_states.values()}

# Get city names from geonamescache and map to their countries
cities = gc.get_cities()
city_country_map = {
    city['name']: countries[city['countrycode']]['name'] for city in cities.values()}

# Define a mapping for common variations
country_variations = {
    "México": "Mexico",
    "Brasil": "Brazil",
    "Türkiye": "Turkey",
    "USA": "United States",
    "U.S.A.": "United States",
    "America": "United States",
    "United States of America": "United States",
    "UK": "United Kingdom",
    "U.K.": "United Kingdom",
    "England": "United Kingdom",
    "Scotland": "United Kingdom",
    "Wales": "United Kingdom",
    "Northern Ireland": "United Kingdom",
    "South Korea": "Korea, Republic of",
    "UAE": "United Arab Emirates",
    "Italia": "Italy",
    "Republic of Korea": "Korea, Republic of",
    "Korea": "Korea, Republic of",
    "North Korea": "Korea, Democratic People's Republic of",
    "Russia": "Russian Federation",
    "Iran": "Islamic Republic of Iran",
}

# Function to extract countries from the affiliation text


def extract_countries(affiliation):
    found_countries = set()

    # Check for country names
    for country in country_names:
        if re.search(r'\b' + re.escape(country) + r'\b', affiliation, re.IGNORECASE):
            found_countries.add(country)

    # Check for country variations
    if len(found_countries) == 0:
        for variation, standard_country in country_variations.items():
            if re.search(r'\b' + re.escape(variation) + r'\b', affiliation, re.IGNORECASE):
                found_countries.add(standard_country)

    # Check for US states
    if len(found_countries) == 0:
        for state in us_state_names:
            if re.search(r'\b' + re.escape(state) + r'\b', affiliation, re.IGNORECASE):
                found_countries.add("United States")
                break

    # Check for city names and map to countries
    if len(found_countries) == 0:
        for city, country in city_country_map.items():
            if re.search(r'\b' + re.escape(city) + r'\b', affiliation, re.IGNORECASE):
                found_countries.add(country)
                break

    return found_countries


# Example DataFrame usage (assuming unique_affiliations_df exists)
unique_affiliations_df['tag_countries'] = unique_affiliations_df['Affiliation'].apply(
    extract_countries)


/var/folders/gx/4hmfp0zd12bblv6znjwbmv600000gn/T/ipykernel_56137/1539185840.py:79: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unique_affiliations_df['tag_countries'] = unique_affiliations_df['Affiliation'].apply(


In [ ]:
unique_affiliations_df.to_csv(
    './data/countries_affiliation_search_approach_2025.csv', index=False)


In [ ]:
unique_affiliations_df


,PMID,LastName,ForeName,Initials,Affiliation,tag_countries
0,35108088,Dai,Fei,F,School of Big Data and Intelligent Engineering...,{China}
3,35108088,Mo,Qi,Q,"School of Software, Yunnan University, Kunming...",{China}
5,35143339,Gurusamy,Rajalakshmi,R,"Department of Information Technology, Sethu In...",{India}
7,35271383,Mara,Alexandru Cristian,AC,Department of Electronics and Information Syst...,{Belgium}
10,35385378,Abidin,Nihan,N,Clinic of Physical Medicine and Rehabilitation...,{Turkey}
...,...,...,...,...,...,...
361930,40455154,Acharya,Vishal,V,Academy of Scientific and Innovative Research ...,{India}
361931,40455159,Bhargava,Harshita,H,"Department of Computer Science and IT, IIS (De...",{India}
361932,40455159,Bhargava,Harshita,H,"Bioclues.org, Pune, India.",{India}
361935,40455159,Valadi,Jayaraman K,JK,"Department of Computer Science, FLAME Universi...",{India}


In [ ]:
grouped_by_pmid = unique_affiliations_df.groupby(
    'PMID')['tag_countries'].apply(list).reset_index(name='tags')
grouped_by_pmid


,PMID,tags
0,35108088,"[{China}, {China}]"
1,35143339,[{India}]
2,35271383,[{Belgium}]
3,35385378,[{Turkey}]
4,35534328,"[{China}, {China}, {China}, {China}]"
...,...,...
39779,40454687,"[{China}, {China}, {China, Macao}, {China}]"
39780,40455017,"[{Russia}, {Russia}]"
39781,40455151,"[{India}, {France}, {India}]"
39782,40455154,"[{India}, {India}, {India}]"


In [ ]:
df_with_country = pd.merge(
    filtered_df_2025, grouped_by_pmid, on='PMID', how='left')
df_with_country


,PMID,MedlineCitationOwner,MedlineCitationStatus,MedlineCitationVersionID,MedlineCitationVersionDate,MedlineCitationIndexingMethod,DateCompleted,DateRevised,ArticlePubModel,JournalISSN,...,GrantList,PublicationTypeList,ArticleDates,PublicationStatus,ArticleIds,History,Objects,Abstract,VernacularTitle,tags
0,35108088,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '10', 'Day': '17'}","{'Year': '2025', 'Month': '04', 'Day': '30'}",Print-Electronic,2167-647X,...,[],"[Journal Article, Research Support, Non-U.S. G...","[{'Year': '2022', 'Month': '02', 'Day': '02', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '35108088'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Tra...",NaN,"[{China}, {China}]"
1,35143339,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '12', 'Day': '16'}","{'Year': '2025', 'Month': '04', 'Day': '30'}",Print-Electronic,2167-647X,...,[],[Journal Article],"[{'Year': '2022', 'Month': '02', 'Day': '10', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '35143339'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Dee...",NaN,[{India}]
2,35271383,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '12', 'Day': '16'}","{'Year': '2025', 'Month': '04', 'Day': '30'}",Print-Electronic,2167-647X,...,[],"[Journal Article, Research Support, Non-U.S. G...","[{'Year': '2022', 'Month': '03', 'Day': '10', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '35271383'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Net...",NaN,[{Belgium}]
3,35385378,NLM,MEDLINE,,,Curated,"{'Year': '2024', 'Month': '10', 'Day': '23'}","{'Year': '2024', 'Month': '11', 'Day': '14'}",Print-Electronic,1949-3614,...,[],"[Journal Article, Controlled Clinical Trial]","[{'Year': '2022', 'Month': '05', 'Day': '05', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '35385378'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'The...",NaN,[{Turkey}]
4,35486562,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '08', 'Day': '08'}","{'Year': '2025', 'Month': '01', 'Day': '07'}",Print-Electronic,1557-9964,...,"[{'GrantID': 'U01 AG024904', 'Acronym': 'AG', ...","[Journal Article, Research Support, U.S. Gov't...","[{'Year': '2024', 'Month': '08', 'Day': '08', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '35486562'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Alz...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42649,40455017,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '06', 'Day': '02'}","{'Year': '2025', 'Month': '06', 'Day': '02'}",Print,0023-1207,...,[],"[English Abstract, Journal Article]",[],ppublish,"[{'IdType': 'pubmed', 'Value': '40455017'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': 'OBJECTIVE', 'NlmCategory': 'OBJECT...",Prognosticheskaya model' i kal'kulyator otsenk...,"[{Russia}, {Russia}]"
42650,40455151,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '06', 'Day': '02'}","{'Year': '2025', 'Month': '06', 'Day': '02'}",Print,1940-6029,...,[],[Journal Article],[],ppublish,"[{'IdType': 'pubmed', 'Value': '40455151'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Vir...",NaN,"[{India}, {France}, {India}]"
42651,40455152,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '06', 'Day': '02'}","{'Year': '2025', 'Month': '06', 'Day': '02'}",Print,1940-6029,...,[],[Journal Article],[],ppublish,"[{'IdType': 'pubmed', 'Value': '40455152'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Und...",NaN,NaN
42652,40455154,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '06', 'Day': '02'}","{'Year': '2025', 'Month': '06', 'Day': '02'}",Print,1940-6029,...,[],[Journal Article]

In [ ]:
df_with_country.to_csv(
    './data/countries_affiliation_df_with_country.csv', index=False)


In [ ]:
# type(df_with_country.iloc[0]['tags'])
canada_df = df_with_country[df_with_country['tags'].apply(
    lambda x: isinstance(x, list) and {'Canada'} in x)]
canada_df


,PMID,MedlineCitationOwner,MedlineCitationStatus,MedlineCitationVersionID,MedlineCitationVersionDate,MedlineCitationIndexingMethod,DateCompleted,DateRevised,ArticlePubModel,JournalISSN,...,GrantList,PublicationTypeList,ArticleDates,PublicationStatus,ArticleIds,History,Objects,Abstract,VernacularTitle,tags
99,37310046,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '10', 'Day': '25'}","{'Year': '2024', 'Month': '10', 'Day': '25'}",Print-Electronic,1521-4095,...,"[{'GrantID': 'K99 AG065495', 'Acronym': 'AG', ...",[Journal Article],"[{'Year': '2023', 'Month': '07', 'Day': '19', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '37310046'}, {'...","[{'PubStatus': 'revised', 'Year': '2023', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Sof...",NaN,"[{United States}, {United States}, {United Sta..."
227,38019000,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '01', 'Day': '09'}","{'Year': '2025', 'Month': '01', 'Day': '09'}",Print-Electronic,1746-1553,...,"[{'GrantID': 'RGPIN-2018-04151', 'Acronym': No...",[Journal Article],"[{'Year': '2023', 'Month': '11', 'Day': '29', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38019000'}, {'...","[{'PubStatus': 'received', 'Year': '2022', 'Mo...",[],"[{'Label': 'BACKGROUND', 'NlmCategory': 'BACKG...",NaN,"[{Canada}, {Canada}, {Canada}, {Canada}]"
268,38190258,NLM,MEDLINE,,,Curated,"{'Year': '2025', 'Month': '01', 'Day': '15'}","{'Year': '2025', 'Month': '05', 'Day': '13'}",Print-Electronic,1949-3614,...,[],"[Journal Article, Randomized Controlled Trial]","[{'Year': '2024', 'Month': '01', 'Day': '08', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38190258'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Chi...",NaN,[{Canada}]
297,38263349,NLM,MEDLINE,,,Automated,"{'Year': '2024', 'Month': '10', 'Day': '08'}","{'Year': '2025', 'Month': '02', 'Day': '20'}",Print-Electronic,1543-0154,...,[],[Editorial],"[{'Year': '2024', 'Month': '01', 'Day': '23', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38263349'}, {'...","[{'PubStatus': 'accepted', 'Year': '2024', 'Mo...",[],NaN,NaN,"[{Canada}, {Canada}, {Canada}, {Canada}, {Cana..."
325,38326673,NLM,MEDLINE,,,Curated,"{'Year': '2025', 'Month': '02', 'Day': '03'}","{'Year': '2025', 'Month': '05', 'Day': '27'}",Print-Electronic,1532-7558,...,"[{'GrantID': 'RG 5144A6/1', 'Acronym': None, '...","[Journal Article, Randomized Controlled Trial]","[{'Year': '2024', 'Month': '02', 'Day': '07', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38326673'}, {'...","[{'PubStatus': 'accepted', 'Year': '2024', 'Mo...",[],"[{'Label': 'BACKGROUND', 'NlmCategory': 'BACKG...",NaN,"[{United States}, {United States}, {United Sta..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42512,40435158,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '05', 'Day': '28'}","{'Year': '2025', 'Month': '05', 'Day': '31'}",Electronic-eCollection,1932-6203,...,[],[Journal Article],"[{'Year': '2025', 'Month': '05', 'Day': '28', ...",epublish,"[{'IdType': 'pubmed', 'Value': '40435158'}, {'...","[{'PubStatus': 'received', 'Year': '2025', 'Mo...",[],"[{'Label': 'PURPOSE', 'NlmCategory': 'OBJECTIV...",NaN,"[{Canada}, {Canada}, {Canada}]"
42557,40440093,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '05', 'Day': '29'}","{'Year': '2025', 'Month': '06', 'Day': '01'}",Print,2047-217X,...,"[{'GrantID': '01ZZ2004', 'Acronym': None, 'Age...",[Journal Article],[],ppublish,"[{'IdType': 'pubmed', 'Value': '40440093'}, {'...","[{'PubStatus': 'received', 'Year': '2024', 'Mo...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Uns...",NaN,"[{United States}, {Russia}, {United States}, {..."
42560,40440289,NLM,MEDLINE,,,Automated,"{'Year': '2025', 'Month': '05', 'Day': '29'}","{'Year': '2025', 'Month': '06', 'Day': '01'}",Electronic-eCollection,1932-6203,...,[],[Journal Article],"[{'Year': '2025', 'Month': '05', 'Day': '29', ...",epublish,"[{'IdType': 'pubmed', 'Value': '40440289'}, {'...","[{'PubS

In [ ]:
canada_df.to_csv(
    './data/countries_affiliation_df_canada_2025.csv', index=False)


In [ ]:
import pandas as pd

# Replace 'your_file.csv' with the path to your CSV file
canada_df = pd.read_csv('./data/countries_affiliation_df_canada_2025.csv')

# Display the first few rows
canada_df


,PMID,MedlineCitationOwner,MedlineCitationStatus,MedlineCitationVersionID,MedlineCitationVersionDate,MedlineCitationIndexingMethod,DateCompleted,DateRevised,ArticlePubModel,JournalISSN,...,GrantList,PublicationTypeList,ArticleDates,PublicationStatus,ArticleIds,History,Objects,Abstract,VernacularTitle,tags
0,37310046,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '10', 'Day': '25'}","{'Year': '2024', 'Month': '10', 'Day': '25'}",Print-Electronic,1521-4095,...,"[{'GrantID': 'K99 AG065495', 'Acronym': 'AG', ...",['Journal Article'],"[{'Year': '2023', 'Month': '07', 'Day': '19', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '37310046'}, {'...","[{'PubStatus': 'revised', 'Year': '2023', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Sof...",NaN,"[{'United States'}, {'United States'}, {'Unite..."
1,38019000,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2025', 'Month': '01', 'Day': '09'}","{'Year': '2025', 'Month': '01', 'Day': '09'}",Print-Electronic,1746-1553,...,"[{'GrantID': 'RGPIN-2018-04151', 'Acronym': No...",['Journal Article'],"[{'Year': '2023', 'Month': '11', 'Day': '29', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38019000'}, {'...","[{'PubStatus': 'received', 'Year': '2022', 'Mo...",[],"[{'Label': 'BACKGROUND', 'NlmCategory': 'BACKG...",NaN,"[{'Canada'}, {'Canada'}, {'Canada'}, {'Canada'}]"
2,38190258,NLM,MEDLINE,NaN,NaN,Curated,"{'Year': '2025', 'Month': '01', 'Day': '15'}","{'Year': '2025', 'Month': '05', 'Day': '13'}",Print-Electronic,1949-3614,...,[],"['Journal Article', 'Randomized Controlled Tri...","[{'Year': '2024', 'Month': '01', 'Day': '08', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38190258'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': ""Chi...",NaN,[{'Canada'}]
3,38263349,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '10', 'Day': '08'}","{'Year': '2025', 'Month': '02', 'Day': '20'}",Print-Electronic,1543-0154,...,[],['Editorial'],"[{'Year': '2024', 'Month': '01', 'Day': '23', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38263349'}, {'...","[{'PubStatus': 'accepted', 'Year': '2024', 'Mo...",[],NaN,NaN,"[{'Canada'}, {'Canada'}, {'Canada'}, {'Canada'..."
4,38326673,NLM,MEDLINE,NaN,NaN,Curated,"{'Year': '2025', 'Month': '02', 'Day': '03'}","{'Year': '2025', 'Month': '05', 'Day': '27'}",Print-Electronic,1532-7558,...,"[{'GrantID': 'RG 5144A6/1', 'Acronym': None, '...","['Journal Article', 'Randomized Controlled Tri...","[{'Year': '2024', 'Month': '02', 'Day': '07', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38326673'}, {'...","[{'PubStatus': 'accepted', 'Year': '2024', 'Mo...",[],"[{'Label': 'BACKGROUND', 'NlmCategory': 'BACKG...",NaN,"[{'United States'}, {'United States'}, {'Unite..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1476,40435158,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2025', 'Month': '05', 'Day': '28'}","{'Year': '2025', 'Month': '05', 'Day': '31'}",Electronic-eCollection,1932-6203,...,[],['Journal Article'],"[{'Year': '2025', 'Month': '05', 'Day': '28', ...",epublish,"[{'IdType': 'pubmed', 'Value': '40435158'}, {'...","[{'PubStatus': 'received', 'Year': '2025', 'Mo...",[],"[{'Label': 'PURPOSE', 'NlmCategory': 'OBJECTIV...",NaN,"[{'Canada'}, {'Canada'}, {'Canada'}]"
1477,40440093,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2025', 'Month': '05', 'Day': '29'}","{'Year': '2025', 'Month': '06', 'Day': '01'}",Print,2047-217X,...,"[{'GrantID': '01ZZ2004', 'Acronym': None, 'Age...",['Journal Article'],[],ppublish,"[{'IdType': 'pubmed', 'Value': '40440093'}, {'...","[{'PubStatus': 'received', 'Year': '2024', 'Mo...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Uns...",NaN,"[{'United States'}, {'Russia'}, {'United State..."
1478,40440289,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2025', 'Month': '05', 'Day': '29'}","{'Year': '2025', 'Month': '06', 'Day': '01'}",Electronic-eCollection,1932-6203,...,[],['Journal Article'],"[{'Year': '2025', 'Month': '05', 'Day': '29', ...

In [ ]:
import pandas as pd

# Replace 'your_file.csv' with the path to your CSV file
gpt_canada_df_2025 = pd.read_csv(
    './data/Patient_Level_Data_Extracted_2025.csv')

# Display the first few rows
gpt_canada_df_2025


,PMID,PLD,Modality
0,37310046,False,NaN
1,38019000,False,NaN
2,38190258,False,NaN
3,38326673,True,"""Clinical"", ""Sensor"", ""Behavioral"", ""Demographic"""
4,38357879,True,Clinical + Sensorimotor (wearable exoskeleton)
...,...,...,...
1425,40435158,False,NaN
1426,40440093,False,NaN
1427,40440289,False,NaN
1428,40448324,False,NaN


In [ ]:
merged_df = pd.merge(canada_df, gpt_canada_df_2025, on='PMID', how='left')

filtered_df = merged_df[merged_df["PLD"] == True]
filtered_df


,PMID,MedlineCitationOwner,MedlineCitationStatus,MedlineCitationVersionID,MedlineCitationVersionDate,MedlineCitationIndexingMethod,DateCompleted,DateRevised,ArticlePubModel,JournalISSN,...,ArticleDates,PublicationStatus,ArticleIds,History,Objects,Abstract,VernacularTitle,tags,PLD,Modality
4,38326673,NLM,MEDLINE,NaN,NaN,Curated,"{'Year': '2025', 'Month': '02', 'Day': '03'}","{'Year': '2025', 'Month': '05', 'Day': '27'}",Print-Electronic,1532-7558,...,"[{'Year': '2024', 'Month': '02', 'Day': '07', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38326673'}, {'...","[{'PubStatus': 'accepted', 'Year': '2024', 'Mo...",[],"[{'Label': 'BACKGROUND', 'NlmCategory': 'BACKG...",NaN,"[{'United States'}, {'United States'}, {'Unite...",True,"""Clinical"", ""Sensor"", ""Behavioral"", ""Demographic"""
5,38357879,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '12', 'Day': '10'}","{'Year': '2025', 'Month': '01', 'Day': '03'}",Print-Electronic,1464-5165,...,"[{'Year': '2024', 'Month': '02', 'Day': '15', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38357879'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': 'PURPOSE', 'NlmCategory': 'UNASSIGN...",NaN,"[{'Canada'}, {'Canada'}, {'Canada'}, {'Canada'...",True,Clinical + Sensorimotor (wearable exoskeleton)
7,38383805,NLM,MEDLINE,NaN,NaN,Manual,"{'Year': '2025', 'Month': '03', 'Day': '13'}","{'Year': '2025', 'Month': '03', 'Day': '13'}",Print-Electronic,2948-2933,...,"[{'Year': '2024', 'Month': '02', 'Day': '21', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38383805'}, {'...","[{'PubStatus': 'received', 'Year': '2023', 'Mo...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'The...",NaN,"[{'Canada'}, {'United States'}, {'United State...",True,"""Imaging"", ""Clinical"""
8,38405768,NLM,PubMed-not-MEDLINE,NaN,NaN,NaN,NaN,"{'Year': '2025', 'Month': '05', 'Day': '30'}",Electronic,NaN,...,"[{'Year': '2024', 'Month': '09', 'Day': '17', ...",epublish,"[{'IdType': 'pubmed', 'Value': '38405768'}, {'...","[{'PubStatus': 'pubmed', 'Year': '2024', 'Mont...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Bip...",NaN,"[{'United States'}, {'United States'}, {'Unite...",True,"""Genomic"", ""Clinical (Phenotype)"""
10,38512435,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '09', 'Day': '17'}","{'Year': '2024', 'Month': '09', 'Day': '17'}",Print-Electronic,2662-4737,...,"[{'Year': '2024', 'Month': '03', 'Day': '21', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38512435'}, {'...","[{'PubStatus': 'received', 'Year': '2023', 'Mo...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Man...",NaN,"[{'Canada'}, {'Switzerland'}, {'Korea, Republi...",True,"""Clinical"", ""Pet"""
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,40270591,NLM,PubMed-not-MEDLINE,NaN,NaN,NaN,NaN,"{'Year': '2025', 'Month': '04', 'Day': '25'}",Electronic-eCollection,2296-889X,...,"[{'Year': '2025', 'Month': '04', 'Day': '09', ...",epublish,"[{'IdType': 'pubmed', 'Value': '40270591'}, {'...","[{'PubStatus': 'received', 'Year': '2025', 'Mo...",[],"[{'Label': '', 'NlmCategory': '', 'Text': ""Thi...",NaN,"[{'Canada'}, {'Canada'}, {'Norway'}, {'Germany'}]",True,Metabolomics + Genomics + XAI (Down syndrome b...
1455,40270958,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2025', 'Month': '04', 'Day': '24'}","{'Year': '2025', 'Month': '04', 'Day': '25'}",Electronic-eCollection,1664-3224,...,"[{'Year': '2025', 'Month': '04', 'Day': '09', ...",epublish,"[{'IdType': 'pubmed', 'Value': '40270958'}, {'...","[{'PubStatus': 'received', 'Year': '2024', 'Mo...",[],"[{'Label': 'INTRODUCTION', 'NlmCategory': 'UNA...",NaN,"[{'Brazil'}, {'Brazil'}, {'Brazil'}, {'Brazil'...",True,Transcriptomic + ML (blood interferome signatu...
1473,40407404,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2025', 'Month': '05', 'Day': '23'}","{'Year': '2025', 'Month': '05', 'Day': '26'}",Print-Electronic,1526-2359,...,"[{'Year': '2025', 'Month': '05', 'Day': '23', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '4040

In [ ]:
# Normalize tag entries (case-insensitive comparison)
def extract_normalized_countries(tag_str):
    if pd.isna(tag_str):
        return set()
    cleaned = tag_str.replace("{", "").replace(
        "}", "").replace("'", "").replace('"', "").replace("[", "").replace(
        "]", "")
    parts = [p.strip().lower()
             for p in cleaned.split(",") if p.strip()]
    return set(parts)


# Apply normalization
filtered_df["UniqueCountries"] = filtered_df["tags"].apply(
    extract_normalized_countries)

# Define conditions
only_canada = filtered_df[filtered_df["UniqueCountries"] == {"canada"}]
collaborative = filtered_df[(filtered_df["UniqueCountries"].apply(lambda x: "canada" in x)) &
                            (filtered_df["UniqueCountries"].apply(lambda x: len(x) > 1))]

# Count results
summary = pd.DataFrame({
    "Category": ["Only Canada", "Collaborative (Canada + Others)"],
    "Count": [len(only_canada), len(collaborative)]
})

summary


/var/folders/gx/4hmfp0zd12bblv6znjwbmv600000gn/T/ipykernel_59767/966200597.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["UniqueCountries"] = filtered_df["tags"].apply(


,Category,Count
0,Only Canada,133
1,Collaborative (Canada + Others),284


In [ ]:
only_canada


,PMID,MedlineCitationOwner,MedlineCitationStatus,MedlineCitationVersionID,MedlineCitationVersionDate,MedlineCitationIndexingMethod,DateCompleted,DateRevised,ArticlePubModel,JournalISSN,...,PublicationStatus,ArticleIds,History,Objects,Abstract,VernacularTitle,tags,PLD,Modality,UniqueCountries
5,38357879,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '12', 'Day': '10'}","{'Year': '2025', 'Month': '01', 'Day': '03'}",Print-Electronic,1464-5165,...,ppublish,"[{'IdType': 'pubmed', 'Value': '38357879'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': 'PURPOSE', 'NlmCategory': 'UNASSIGN...",NaN,"[{'Canada'}, {'Canada'}, {'Canada'}, {'Canada'...",True,Clinical + Sensorimotor (wearable exoskeleton),{canada}
14,38627132,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '10', 'Day': '19'}","{'Year': '2025', 'Month': '03', 'Day': '20'}",Print-Electronic,1878-4046,...,ppublish,"[{'IdType': 'pubmed', 'Value': '38627132'}, {'...","[{'PubStatus': 'received', 'Year': '2023', 'Mo...",[],"[{'Label': 'RATIONALE', 'NlmCategory': '', 'Te...",NaN,"[{'Canada'}, {'Canada'}, {'Canada'}, {'Canada'}]",True,"""Imaging"", ""Clinical"", ""Demographic"", CT images",{canada}
15,38647191,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '07', 'Day': '22'}","{'Year': '2024', 'Month': '07', 'Day': '22'}",Print-Electronic,1522-2594,...,ppublish,"[{'IdType': 'pubmed', 'Value': '38647191'}, {'...","[{'PubStatus': 'revised', 'Year': '2024', 'Mon...",[],"[{'Label': 'PURPOSE', 'NlmCategory': 'OBJECTIV...",NaN,"[{'Canada'}, {'Canada'}, {'Canada'}, {'Canada'...",True,"""Clinical"", ""Sensor"", ""Demographic"", ""Cognitiv...",{canada}
29,38905892,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '07', 'Day': '23'}","{'Year': '2024', 'Month': '07', 'Day': '23'}",Print-Electronic,1879-0534,...,ppublish,"[{'IdType': 'pubmed', 'Value': '38905892'}, {'...","[{'PubStatus': 'received', 'Year': '2024', 'Mo...",[],"[{'Label': 'INTRODUCTION', 'NlmCategory': 'BAC...",NaN,"[{'Canada'}, {'Canada'}, {'Canada'}]",True,Imaging (knee MRI segmentation),{canada}
39,38981593,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '07', 'Day': '19'}","{'Year': '2024', 'Month': '07', 'Day': '19'}",Electronic,1361-6560,...,epublish,"[{'IdType': 'pubmed', 'Value': '38981593'}, {'...","[{'PubStatus': 'received', 'Year': '2024', 'Mo...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Obj...",NaN,"[{'Canada'}, {'Canada'}]",True,Imaging (MRI + CT for GAN-based CT synthesis),{canada}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1422,40091888,NLM,PubMed-not-MEDLINE,NaN,NaN,NaN,NaN,"{'Year': '2025', 'Month': '03', 'Day': '18'}",Electronic-eCollection,2054-3581,...,epublish,"[{'IdType': 'pubmed', 'Value': '40091888'}, {'...","[{'PubStatus': 'received', 'Year': '2024', 'Mo...",[],"[{'Label': 'BACKGROUND', 'NlmCategory': 'UNASS...",NaN,"[{'Canada'}, {'Canada'}, {'Canada'}]",True,Clinical + Imaging + Tabular (kidney transplan...,{canada}
1427,40103927,NLM,PubMed-not-MEDLINE,NaN,NaN,NaN,NaN,"{'Year': '2025', 'Month': '03', 'Day': '20'}",Electronic-eCollection,1663-4365,...,epublish,"[{'IdType': 'pubmed', 'Value': '40103927'}, {'...","[{'PubStatus': 'received', 'Year': '2024', 'Mo...",[],"[{'Label': 'BACKGROUND', 'NlmCategory': 'UNASS...",NaN,"[{'Canada'}, {'Canada'}]",True,"Clinical (multi-modal biomarkers, cognitive)",{canada}
1438,40160858,NLM,PubMed-not-MEDLINE,NaN,NaN,NaN,NaN,"{'Year': '2025', 'Month': '04', 'Day': '02'}",Electronic-eCollection,2001-0370,...,epublish,"[{'IdType': 'pubmed', 'Value': '40160858'}, {'...","[{'PubStatus': 'received', 'Year': '2024', 'Mo...",[],"[{'Label': '', 'NlmCategory': '', 'Text': ""Und...",NaN,"[{'Canada'}, {'Canada'}, {'Canada'}, {'Canada'...",True,Clinical + Imaging (metabolomics + XAI in hear...,{canada}
1445,40182983,NLM,PubMed-not-MEDLINE,NaN,NaN,NaN,NaN,"{'Year': '2025', 'Month': '04', 'Day': '05'}",Electronic-eCollection,2666-9145,...,epublish,"[{'IdType': 'pubme

In [ ]:
collaborative


,PMID,MedlineCitationOwner,MedlineCitationStatus,MedlineCitationVersionID,MedlineCitationVersionDate,MedlineCitationIndexingMethod,DateCompleted,DateRevised,ArticlePubModel,JournalISSN,...,PublicationStatus,ArticleIds,History,Objects,Abstract,VernacularTitle,tags,PLD,Modality,UniqueCountries
4,38326673,NLM,MEDLINE,NaN,NaN,Curated,"{'Year': '2025', 'Month': '02', 'Day': '03'}","{'Year': '2025', 'Month': '05', 'Day': '27'}",Print-Electronic,1532-7558,...,ppublish,"[{'IdType': 'pubmed', 'Value': '38326673'}, {'...","[{'PubStatus': 'accepted', 'Year': '2024', 'Mo...",[],"[{'Label': 'BACKGROUND', 'NlmCategory': 'BACKG...",NaN,"[{'United States'}, {'United States'}, {'Unite...",True,"""Clinical"", ""Sensor"", ""Behavioral"", ""Demographic""","{united states, canada}"
7,38383805,NLM,MEDLINE,NaN,NaN,Manual,"{'Year': '2025', 'Month': '03', 'Day': '13'}","{'Year': '2025', 'Month': '03', 'Day': '13'}",Print-Electronic,2948-2933,...,ppublish,"[{'IdType': 'pubmed', 'Value': '38383805'}, {'...","[{'PubStatus': 'received', 'Year': '2023', 'Mo...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'The...",NaN,"[{'Canada'}, {'United States'}, {'United State...",True,"""Imaging"", ""Clinical""","{united states, canada}"
8,38405768,NLM,PubMed-not-MEDLINE,NaN,NaN,NaN,NaN,"{'Year': '2025', 'Month': '05', 'Day': '30'}",Electronic,NaN,...,epublish,"[{'IdType': 'pubmed', 'Value': '38405768'}, {'...","[{'PubStatus': 'pubmed', 'Year': '2024', 'Mont...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Bip...",NaN,"[{'United States'}, {'United States'}, {'Unite...",True,"""Genomic"", ""Clinical (Phenotype)""","{united states, greece, republic of, sweden, s..."
10,38512435,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '09', 'Day': '17'}","{'Year': '2024', 'Month': '09', 'Day': '17'}",Print-Electronic,2662-4737,...,ppublish,"[{'IdType': 'pubmed', 'Value': '38512435'}, {'...","[{'PubStatus': 'received', 'Year': '2023', 'Mo...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Man...",NaN,"[{'Canada'}, {'Switzerland'}, {'Korea, Republi...",True,"""Clinical"", ""Pet""","{hungary, canada, switzerland, republic of, de..."
18,38789645,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '07', 'Day': '19'}","{'Year': '2024', 'Month': '07', 'Day': '27'}",Print-Electronic,1546-170X,...,ppublish,"[{'IdType': 'pubmed', 'Value': '38789645'}, {'...","[{'PubStatus': 'received', 'Year': '2023', 'Mo...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Pre...",NaN,"[{'Netherlands'}, {'Netherlands'}, {'Switzerla...",True,"""Imaging (Histopathology Whole-Slide Images)"",...","{australia, canada, switzerland, united kingdo..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,40270591,NLM,PubMed-not-MEDLINE,NaN,NaN,NaN,NaN,"{'Year': '2025', 'Month': '04', 'Day': '25'}",Electronic-eCollection,2296-889X,...,epublish,"[{'IdType': 'pubmed', 'Value': '40270591'}, {'...","[{'PubStatus': 'received', 'Year': '2025', 'Mo...",[],"[{'Label': '', 'NlmCategory': '', 'Text': ""Thi...",NaN,"[{'Canada'}, {'Canada'}, {'Norway'}, {'Germany'}]",True,Metabolomics + Genomics + XAI (Down syndrome b...,"{norway, canada, germany}"
1455,40270958,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2025', 'Month': '04', 'Day': '24'}","{'Year': '2025', 'Month': '04', 'Day': '25'}",Electronic-eCollection,1664-3224,...,epublish,"[{'IdType': 'pubmed', 'Value': '40270958'}, {'...","[{'PubStatus': 'received', 'Year': '2024', 'Mo...",[],"[{'Label': 'INTRODUCTION', 'NlmCategory': 'UNA...",NaN,"[{'Brazil'}, {'Brazil'}, {'Brazil'}, {'Brazil'...",True,Transcriptomic + ML (blood interferome signatu...,"{united states, brazil, canada}"
1473,40407404,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2025', 'Month': '05', 'Day': '23'}","{'Year': '2025', 'Month': '05', 'Day': '26'}",Print-Electronic,1526-2359,...,ppublish,"[{'IdType': 'pubmed', 'Value': '40407404'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': ""Int...",NaN,"[{'Turk

In [ ]:
import pandas as pd

# Replace 'your_file.csv' with the path to your CSV file
data_canada_map_modalities = pd.read_csv(
    './data-canada-map-modalities.csv')

# Display the first few rows
data_canada_map_modalities


,pid,combined_modalities
0,28987903,"{""[\""Cardiac Images (MR sequences)\""]"",""[\""MR\..."
1,29065782,"{""[\""Clinical\""]"",""Medicine\""]"",""[\""Surgery""}"
2,29071211,"{""[\""Myelin"",""[\""Myelin Maps"",""T1-weighted MRI..."
3,29084099,"{""Executive Function\""]"",""[\""Robotics"",""[\""Sen..."
4,29169029,"{CT,""CT images"",""[\""Electron Microscopy"",""MRI\..."
...,...,...
972,38962394,"{""[{\""Modality_Name\"": \""Speech\""}, {\""Modalit..."
973,38977181,"{""Medical\""]"",""[\""Sociodemographic"",""[\""Text\""]""}"
974,38980984,"{""accelerometry\""]"",""Accelerometry\""]"",""[\""ele..."
975,38987512,"{""[\""3D spinal ultrasonographs\""]"",""[\""Ultraso..."


In [ ]:
canada_df = canada_df[~canada_df['PublicationTypeList'].str.contains(
    'Editorial', na=False)]

canada_df


,PMID,MedlineCitationOwner,MedlineCitationStatus,MedlineCitationVersionID,MedlineCitationVersionDate,MedlineCitationIndexingMethod,DateCompleted,DateRevised,ArticlePubModel,JournalISSN,...,GrantList,PublicationTypeList,ArticleDates,PublicationStatus,ArticleIds,History,Objects,Abstract,VernacularTitle,tags
0,37310046,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '10', 'Day': '25'}","{'Year': '2024', 'Month': '10', 'Day': '25'}",Print-Electronic,1521-4095,...,"[{'GrantID': 'K99 AG065495', 'Acronym': 'AG', ...",['Journal Article'],"[{'Year': '2023', 'Month': '07', 'Day': '19', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '37310046'}, {'...","[{'PubStatus': 'revised', 'Year': '2023', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Sof...",NaN,"[{'United States'}, {'United States'}, {'Unite..."
1,38019000,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2025', 'Month': '01', 'Day': '09'}","{'Year': '2025', 'Month': '01', 'Day': '09'}",Print-Electronic,1746-1553,...,"[{'GrantID': 'RGPIN-2018-04151', 'Acronym': No...",['Journal Article'],"[{'Year': '2023', 'Month': '11', 'Day': '29', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38019000'}, {'...","[{'PubStatus': 'received', 'Year': '2022', 'Mo...",[],"[{'Label': 'BACKGROUND', 'NlmCategory': 'BACKG...",NaN,"[{'Canada'}, {'Canada'}, {'Canada'}, {'Canada'}]"
2,38190258,NLM,MEDLINE,NaN,NaN,Curated,"{'Year': '2025', 'Month': '01', 'Day': '15'}","{'Year': '2025', 'Month': '05', 'Day': '13'}",Print-Electronic,1949-3614,...,[],"['Journal Article', 'Randomized Controlled Tri...","[{'Year': '2024', 'Month': '01', 'Day': '08', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38190258'}, {'...","[{'PubStatus': 'medline', 'Year': '2025', 'Mon...",[],"[{'Label': '', 'NlmCategory': '', 'Text': ""Chi...",NaN,[{'Canada'}]
4,38326673,NLM,MEDLINE,NaN,NaN,Curated,"{'Year': '2025', 'Month': '02', 'Day': '03'}","{'Year': '2025', 'Month': '05', 'Day': '27'}",Print-Electronic,1532-7558,...,"[{'GrantID': 'RG 5144A6/1', 'Acronym': None, '...","['Journal Article', 'Randomized Controlled Tri...","[{'Year': '2024', 'Month': '02', 'Day': '07', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38326673'}, {'...","[{'PubStatus': 'accepted', 'Year': '2024', 'Mo...",[],"[{'Label': 'BACKGROUND', 'NlmCategory': 'BACKG...",NaN,"[{'United States'}, {'United States'}, {'Unite..."
5,38357879,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '12', 'Day': '10'}","{'Year': '2025', 'Month': '01', 'Day': '03'}",Print-Electronic,1464-5165,...,[],['Journal Article'],"[{'Year': '2024', 'Month': '02', 'Day': '15', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '38357879'}, {'...","[{'PubStatus': 'medline', 'Year': '2024', 'Mon...",[],"[{'Label': 'PURPOSE', 'NlmCategory': 'UNASSIGN...",NaN,"[{'Canada'}, {'Canada'}, {'Canada'}, {'Canada'..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1476,40435158,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2025', 'Month': '05', 'Day': '28'}","{'Year': '2025', 'Month': '05', 'Day': '31'}",Electronic-eCollection,1932-6203,...,[],['Journal Article'],"[{'Year': '2025', 'Month': '05', 'Day': '28', ...",epublish,"[{'IdType': 'pubmed', 'Value': '40435158'}, {'...","[{'PubStatus': 'received', 'Year': '2025', 'Mo...",[],"[{'Label': 'PURPOSE', 'NlmCategory': 'OBJECTIV...",NaN,"[{'Canada'}, {'Canada'}, {'Canada'}]"
1477,40440093,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2025', 'Month': '05', 'Day': '29'}","{'Year': '2025', 'Month': '06', 'Day': '01'}",Print,2047-217X,...,"[{'GrantID': '01ZZ2004', 'Acronym': None, 'Age...",['Journal Article'],[],ppublish,"[{'IdType': 'pubmed', 'Value': '40440093'}, {'...","[{'PubStatus': 'received', 'Year': '2024', 'Mo...",[],"[{'Label': '', 'NlmCategory': '', 'Text': 'Uns...",NaN,"[{'United States'}, {'Russia'}, {'United State..."
1478,40440289,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2025', 'Month': '05', 'Day': '29'}","{'Year': '2025', 'Month': '06', 'Day': '01'}",Electronic-eCollection,1932-6203,...,[],['Journal Articl

In [ ]:
import pandas as pd
import ast

# Read the CSV file
# file_path = './data/countries_affiliation_df_canada.csv'  # Replace with your actual file path
# canada_df = pd.read_csv(file_path)

# Convert JSON-like strings to proper Python objects
columns_to_convert = ['DateRevised', 'Abstract', 'Authors',
                      'ArticleIds', 'History', 'tags', 'ArticleDates']

for column in columns_to_convert:
    canada_df[column] = canada_df[column].apply(
        lambda x: ast.literal_eval(x) if pd.notna(x) else x)

# Display the DataFrame
print(canada_df.dtypes)
print(canada_df.head())


PMID                               int64
MedlineCitationOwner              object
MedlineCitationStatus             object
MedlineCitationVersionID         float64
MedlineCitationVersionDate        object
MedlineCitationIndexingMethod     object
DateRevised                       object
ArticlePubModel                   object
JournalISSN                       object
JournalTitle                      object
JournalISOAbbreviation            object
ArticleTitle                      object
Language                          object
StartPage                         object
EndPage                           object
MedlinePgn                        object
Abstract                          object
Authors                           object
DataBankList                      object
GrantList                         object
PublicationTypeList               object
ArticleDates                      object
PublicationStatus                 object
ArticleIds                        object
History         

In [ ]:
type(canada_df.iloc[0]["PMID"])


numpy.int64

In [ ]:
from langchain_community.llms import Ollama
import json
import os
import datetime

llm = Ollama(model="llama3")

# Define the prompt text
prompt_text = """
Consider the following paper details: 
-  Title: 
   {Title}

- Abstract
  {Abstract}

- Affiliations: 
  {Affiliations}

GIVE ME A JSON object should include:
* Affiliation_countries_tags
* Centralized_or_Decentralized
* Is_Real_World_Data_Set
* Size_of_Data_Set
* AI_or_ML_Tools_In_The_Paper
* Governance_Mode
* Patient_Level_Data
* Type_of_Patient_Level_Data
* Clinical_Data_Type
* Is_Using_Survey_Data
* Is_Using_Questionnaire
* Federated_Learning_Techniques
* Privacy_Preserving_Techniques
* Data_Collection_Method
* Is_Using_Large_Language_Model
* Is_Using_Multimodal_Model
* Is_Using_Shared_Dataset
* Modalities

In your response, please do not use abbreviations for any fields. 
Ensure to return only the JSON result without any additional text or commentary.
JSON SHOULD ONLY CONTAIN THE ABOVE FIELD. 
Use "NIL" for any null or unspecified values.
"""

# Define the path for the results file
results_file_path = 'canada_review_results_2025.json'

# Load existing results if the file exists
if os.path.exists(results_file_path):
    with open(results_file_path, 'r') as file:
        existing_results = json.load(file)
else:
    existing_results = {}


# Initialize a list to store the results
canada_review = []

# Define the batch size for saving results
batch_size = 10

# Ensure we only process the first 100 rows
num_rows_to_process = min(len(canada_df), 1500)

# Iterate over the rows to process
for index in range(num_rows_to_process):
    row = canada_df.iloc[index]
    PMID = str(row["PMID"])
    
    # Skip processing if PMID already exists in the results
    if PMID in existing_results:
        print(f"Skipping PMID {PMID} as it already exists in the results.")
        continue

    Title = row["ArticleTitle"]
    Abstract = row["Abstract"]
    Affiliations = row["Authors"]
    
    # Format the prompt with the current details
    prompt_with_item = prompt_text.format(Affiliations=Affiliations, Abstract=Abstract, Title=Title)
    
    # Invoke the LLM
    res = llm.invoke(prompt_with_item, max_tokens=1)
    
    # Append the result to the list
    canada_review.append({"PMID": PMID, "llm_res": res})

    # Add the new result to the existing results dictionary
    existing_results[PMID] = res

    # Write to file after every batch_size rows
    if (index + 1) % batch_size == 0:
        with open(results_file_path, 'w') as file:
            json.dump(existing_results, file, indent=4)
        print(f"{datetime.datetime.now().time().strftime("%Y-%m-%d %H:%M:%S")} Saved results after processing {index + 1} rows.")

# Save any remaining results after the loop
if (num_rows_to_process % batch_size) != 0:
    with open(results_file_path, 'w') as file:
        json.dump(existing_results, file, indent=4)
    print(f"{datetime.datetime.now().time().strftime("%Y-%m-%d %H:%M:%S")} Saved final results after processing {num_rows_to_process} rows.")

# Print the results
for i, row in enumerate(canada_review):
    print(f"Row {i+1}: {row}")


Skipping PMID 37310046 as it already exists in the results.
Skipping PMID 38019000 as it already exists in the results.
Skipping PMID 38190258 as it already exists in the results.
Skipping PMID 38326673 as it already exists in the results.
Skipping PMID 38357879 as it already exists in the results.
Skipping PMID 38383805 as it already exists in the results.
Skipping PMID 38405768 as it already exists in the results.
Skipping PMID 38409223 as it already exists in the results.
Skipping PMID 38512435 as it already exists in the results.
Skipping PMID 38520646 as it already exists in the results.
Skipping PMID 38575195 as it already exists in the results.
Skipping PMID 38590142 as it already exists in the results.
Skipping PMID 38627132 as it already exists in the results.
Skipping PMID 38647191 as it already exists in the results.
Skipping PMID 38696699 as it already exists in the results.
Skipping PMID 38717330 as it already exists in the results.
Skipping PMID 38789645 as it already exi

In [ ]:
# all canadian papers related to AI and ML
canada_df


,PMID,MedlineCitationOwner,MedlineCitationStatus,MedlineCitationVersionID,MedlineCitationVersionDate,MedlineCitationIndexingMethod,DateRevised,ArticlePubModel,JournalISSN,JournalTitle,...,GrantList,PublicationTypeList,ArticleDates,PublicationStatus,ArticleIds,History,Objects,DateCompleted,VernacularTitle,tags
0,28350526,NLM,MEDLINE,NaN,NaN,NaN,"{'Year': '2018', 'Month': '10', 'Day': '29'}",Print-Electronic,1748-3115,Disability and rehabilitation. Assistive techn...,...,[],"['Journal Article', 'Observational Study', ""Re...","[{'Year': '2017', 'Month': '03', 'Day': '28', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '28350526'}, {'...","[{'PubStatus': 'pubmed', 'Year': '2017', 'Mont...",[],"{'Year': '2018', 'Month': '10', 'Day': '29'}",NaN,"[{Canada}, {Canada}]"
1,28444633,NLM,MEDLINE,NaN,NaN,Curated,"{'Year': '2019', 'Month': '12', 'Day': '10'}",Print,1531-5320,Psychonomic bulletin & review,...,"[{'GrantID': 'Discovery Grant (312409-05)', 'A...","['Journal Article', ""Research Support, Non-U.S...",[],ppublish,"[{'IdType': 'pubmed', 'Value': '28444633'}, {'...","[{'PubStatus': 'pubmed', 'Year': '2017', 'Mont...",[],"{'Year': '2019', 'Month': '02', 'Day': '07'}",NaN,"[{Canada}, {Canada}]"
2,28550657,NLM,MEDLINE,NaN,NaN,Curated,"{'Year': '2019', 'Month': '02', 'Day': '15'}",Print,1554-3528,Behavior research methods,...,[],['Journal Article'],[],ppublish,"[{'IdType': 'pubmed', 'Value': '28550657'}, {'...","[{'PubStatus': 'pubmed', 'Year': '2017', 'Mont...",[],"{'Year': '2019', 'Month': '02', 'Day': '11'}",NaN,[{Canada}]
3,28589480,NLM,MEDLINE,NaN,NaN,Curated,"{'Year': '2019', 'Month': '12', 'Day': '10'}",Print,1573-3602,Journal of gambling studies,...,"[{'GrantID': '3441', 'Acronym': None, 'Agency'...",['Journal Article'],[],ppublish,"[{'IdType': 'pubmed', 'Value': '28589480'}, {'...","[{'PubStatus': 'pubmed', 'Year': '2017', 'Mont...",[],"{'Year': '2018', 'Month': '07', 'Day': '16'}",NaN,"[{Canada}, {Canada}, {Canada}, {Canada}]"
4,28596271,NLM,MEDLINE,NaN,NaN,NaN,"{'Year': '2019', 'Month': '10', 'Day': '25'}",Print-Electronic,1522-1601,"Journal of applied physiology (Bethesda, Md. :...",...,[],"['Clinical Trial', 'Journal Article', ""Researc...","[{'Year': '2017', 'Month': '06', 'Day': '08', ...",ppublish,"[{'IdType': 'pubmed', 'Value': '28596271'}, {'...","[{'PubStatus': 'pubmed', 'Year': '2017', 'Mont...",[],"{'Year': '2019', 'Month': '10', 'Day': '25'}",NaN,"[{Canada}, {Brazil}, {Canada}, {Canada}]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7751,39013736,NLM,Publisher,NaN,NaN,Automated,"{'Year': '2024', 'Month': '07', 'Day': '16'}",Print-Electronic,1878-4046,Academic radiology,...,[],['Journal Article'],"[{'Year': '2024', 'Month': '07', 'Day': '15', ...",aheadofprint,"[{'IdType': 'pubmed', 'Value': '39013736'}, {'...","[{'PubStatus': 'received', 'Year': '2024', 'Mo...",[],NaN,NaN,"[{Canada}, {Canada}, {Canada}, {Canada}, {Cana..."
7752,39013848,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '07', 'Day': '16'}",Electronic,2041-1723,Nature communications,...,[],['Journal Article'],"[{'Year': '2024', 'Month': '07', 'Day': '17', ...",epublish,"[{'IdType': 'pubmed', 'Value': '39013848'}, {'...","[{'PubStatus': 'received', 'Year': '2023', 'Mo...",[],"{'Year': '2024', 'Month': '07', 'Day': '16'}",NaN,"[{Germany}, {Japan}, {Germany}, {United States..."
7753,39013867,NLM,MEDLINE,NaN,NaN,Automated,"{'Year': '2024', 'Month': '07', 'Day': '16'}",Electronic,2041-1723,Nature communications,...,"[{'GrantID': 'PJT-180505', 'Acronym': None, 'A...",['Journal Article'],"[{'Year': '2024', 'Month': '07', 'Day': '16', ...",epublish,"[{'IdType': 'pubmed', 'Value': '39013867'}, {'...","[{'PubStatus': 'received', 'Year': '2023', 'Mo...",[],"{'Year': '2024', 'Month': '07', 'Day': '16'}",NaN,"[{Canada}, {Canada}, {Canada}, {Canada}, {Cana..."
7754,39015068,NLM,Publisher,NaN,NaN,Automated,"{'Year': '2024', 'Month': '07', 'Day': '17'}",Print-Electronic,1097-6817,Otolaryngology--head and neck surgery

In [ ]:
import json
import pandas as pd

# Load raw data
with open('canada_review_results_2025.json', 'r') as file:
    raw_data = json.load(file)

# Clean and parse each stringified JSON object
parsed_data = {}
for k, v in raw_data.items():
    # Replace invalid "NIL" with proper JSON null
    cleaned_str = v.replace('NIL', 'null')

    try:
        parsed_data[k] = json.loads(cleaned_str)
    except json.JSONDecodeError as e:
        print(f"Error decoding record {k}: {e}")
        parsed_data[k] = None  # or skip / log for review

# Filter out failed entries
parsed_data = {k: v for k, v in parsed_data.items() if v is not None}

# Convert to DataFrame
df = pd.DataFrame.from_dict(parsed_data, orient='index').reset_index()
df.rename(columns={'index': 'Record_ID'}, inplace=True)

# Display the first few rows
df.head()


Error decoding record 38383805: Expecting value: line 1 column 1 (char 0)
Error decoding record 38405768: Expecting ',' delimiter: line 7 column 22 (char 267)
Error decoding record 38512435: Expecting ',' delimiter: line 10 column 27 (char 413)
Error decoding record 38575195: Expecting ':' delimiter: line 3 column 11 (char 46)
Error decoding record 38647191: Expecting ',' delimiter: line 5 column 25 (char 157)
Error decoding record 38696699: Expecting ',' delimiter: line 3 column 35 (char 113)
Error decoding record 38789645: Expecting ',' delimiter: line 19 column 48 (char 808)
Error decoding record 38857570: Expecting ',' delimiter: line 19 column 32 (char 698)
Error decoding record 38871841: Expecting ',' delimiter: line 4 column 31 (char 152)
Error decoding record 38878395: Expecting ',' delimiter: line 15 column 29 (char 490)
Error decoding record 38880121: Expecting value: line 1 column 1 (char 0)
Error decoding record 38881045: Expecting ',' delimiter: line 7 column 24 (char 226)

,Record_ID,Affiliation_countries_tags,Centralized_or_Decentralized,Is_Real_World_Data_Set,Size_of_Data_Set,AI_or_ML_Tools_In_The_Paper,Governance_Mode,Patient_Level_Data,Type_of_Patient_Level_Data,Clinical_Data_Type,...,C-centralized_or_decentralized,Centalized_or_Decentralized,type of Patient_Level_Data,Real_World_Data_Set,Using_Survey_Data,Using_Questionnaire,Is_Used_Shared_Dataset,type of Patient Level Data,data Collection Method,data collection method
0,37310046,"[USA, Canada]",Decentralized,True,null,null,null,False,null,null,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,38019000,[Canada],Decentralized,True,null,null,null,False,null,null,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,38190258,[Canada],Decentralized,False,null,null,null,False,null,null,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,38409223,"[Canada, USA]",Decentralized,True,over 33 million cells,Generative Pretrained Transformers,null,False,null,null,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,38590142,"[Canada, Canada, Canada]",Decentralized,True,null,"[Ensemble learning methods, Random forest, Gra...",null,False,null,null,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import pandas as pd
import json

# Load the JSON file
file_path = 'data_canada_v1_m1.json'
with open(file_path, 'r') as file:
    data = json.load(file)

# Convert the JSON data to a DataFrame
data_canada_v1_m1 = pd.DataFrame(data)

# Display the first few rows of the DataFrame
data_canada_v1_m1.head()


ValueError: If using all scalar values, you must pass an index

In [ ]:
m1_patient_level_data = data_canada_v1_m1[data_canada_v1_m1['Patient_Level_Data'].astype(
    str).str.lower() == 'true']
len(m1_patient_level_data)


2902

In [ ]:
m1_patient_level_data.iloc[0]["PID"]


'27741066'

In [ ]:
canada_df.iloc[0]['PMID']


AttributeError: 'numpy.int64' object has no attribute 'dtypes'

In [ ]:
import pandas as pd
import json

# Load the JSON file
file_path = 'data_canada_v1_m2.json'
with open(file_path, 'r') as file:
    data = json.load(file)

# Convert the JSON data to a DataFrame
data_canada_v1_m2 = pd.DataFrame(data)

# Display the first few rows of the DataFrame
data_canada_v1_m2.head()


,Affiliation_countries_tags,Centralized_or_Decentralized,Is_Real_World_Data_Set,Size_of_Data_Set,AI_or_ML_Tools_In_The_Paper,Governance_Mode,Patient_Level_Data,Type_of_Patient_Level_Data,Clinical_Data_Type,Is_Using_Survey_Data,...,Is_Using_Genomic_Data,Is_Using_Other_Kinds_of_Data,Is_Private,Is_Public,Is_Open,Is_Semi_Public,Is_Semi_Private,Is_Commercially_Sensitive,Is_Sensitive,Is_Industrially_Sensitive
0,None,Decentralized,True,"{'total': 28341, 'returned': 626}","[Artificial Neural Network, Classification Tree]",None,True,None,Predictive model,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,None,False,None,None,None,True,None (no patient-level data mentioned),None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,[Canada],Decentralized,False,None,True,None,False,None,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Decentralized,False,None,True,None,None,None,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"[{'Country': 'Canada', 'Tag': None}, {'Country...",None,False,None,None,None,False,None,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
m2_patient_level_data = data_canada_v1_m2[data_canada_v1_m2['Patient_Level_Data'].astype(
    str).str.lower() == 'true']
len(m2_patient_level_data)


2517

In [ ]:
import numpy as np

m1_patient_level_data['PID'] = m1_patient_level_data['PID'].astype(np.int64)


/var/folders/gx/4hmfp0zd12bblv6znjwbmv600000gn/T/ipykernel_89240/4087479661.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  m1_patient_level_data['PID'] = m1_patient_level_data['PID'].astype(np.int64)


In [ ]:
m1_patient_level_data.iloc[0]["PID"]


27741066

In [ ]:
canada_df_merged_m1 = pd.merge(
    canada_df, m1_patient_level_data, how='left', left_on='PMID', right_on='PID')


In [ ]:
canada_df_merged_m1.iloc[1]  # ['Patient_Level_Data']


PMID                               28444633
MedlineCitationOwner                    NLM
MedlineCitationStatus               MEDLINE
MedlineCitationVersionID                NaN
MedlineCitationVersionDate              NaN
                                     ...   
Isof_Using_Shared_Dataset               NaN
Isof_Using_Large_Language_Model         NaN
Isof_Using_Multimodal_Model             NaN
Isover_ML_Tools_In_The_Paper            NaN
Impact_on_Radiation_Therapy             NaN
Name: 1, Length: 198, dtype: object

In [ ]:
# Set display options to show all columns and rows
pd.set_option('display.max_columns', None)  # None means no limit
pd.set_option('display.max_rows', None)  # Adjust based on your needs
pd.set_option('display.max_colwidth', None)  # No limit on column width
m1_patient_level_data.head(20)


,Affiliation_countries,Centralized_or_Decentralized,Is_Real_World_Data_Set,Size_of_Data_Set,AI_or_ML_Tools_In_The_Paper,Governance_Mode,Patient_Level_Data,Type_of_Patient_Level_Data,Clinical_Data_Type,Is_Using_Survey_Data,Is_Using_Questionnaire,Federated_Learning_Techniques,Privacy_Preserving_Techniques,Data_Collection_Method,Is_Using_Large_Language_Model,Is_Using_Multimodal_Model,Is_Using_Shared_Dataset,Modalities,PID,Affiliation_countries_tags,type_of_patient_level_data,data_collection_method,is_using_large_language_model,is_using_multimodal_model,is_using_shared_dataset,Countries,data_Collection_Method,is_using_survey_data,is_using_questionnaire,Ai_or_ml_tools_in_the_paper,type of patient level data,Clinical Data Type,is using survey data,is using questionnaire,Federated Learning Techniques,Privacy Preserving Techniques,data collection method,is using large language model,is using multimodal model,is using shared dataset,Ai_or_ML_Tools_In_The_Paper,type of Patient Level Data,modalities,is_Using_Survey_Data,is_Using_Questionnaire,is_Using_Large_Language_Model,is_Using_Multimodal_Model,is_Using_Shared_Dataset,Governance_mode,Patient_level_data,Type_of_patient_level_data,Clinical_data_type,Is_using_survey_data,Is_using_questionnaire,Federated_learning_techniques,Privacy_preserving_techniques,Data_collection_method,Is_using_large_language_model,Is_using_multimodal_model,Is_using_shared_dataset,Countries_tags,Is_Using_Claimed_To_Be_Real_World_Data_Set,Is_Using_Patient_Level_Data,Ai_or_Ml_Tools_In_The_Paper,Countries_tag,type of Patient_Level_Data,Tags,C centralized_or_decen_tralized,is_USing_Survey_Data,is_USing_Questionnaire,is_USing_Large_Language_Model,is_USing_Multimodal_Model,is_USing_Shared_Dataset,Centalized_or_Decentralized,C centralized_or_Decentralized,Iso_of_Data_Set,Isof_Patient_Level_Data,P patient_Level_Data_Type,Centralized,Decentralized,is_using_Survey_Data,Countries_tagged,Impact_of_Privacy_Preserving_Techniques,Is_Using_Real_World_Data_Set,C-centralized_or-Decentralized,Type_of_Patient_Level_Dose,is Using Survey Data,is Using Questionnaire,data Collection Method,is Using Large Language Model,is Using Multimodal Model,is Using Shared Dataset,Data Collection Method,Is_Using LARGE_LANGUAGE_MODEL,Is_Using_MULTIMODAL_MODEL,Is_Using_SHARED_DATASET,Real_world_data_set,Survey_Data,Questionnaire,Largest_Language_Model,Multimodal_Model,Shared_Dataset,Is Using Survey Data,Is Using Questionnaire,Is Using Large Language Model,Is Using Multimodal Model,Is Using Shared Dataset,Real_World_Data_Set,Using_Survey_Data,Using_Questionnaire,Using_Large_Language_Model,Using_Multimodal_Model,Using_Shared_Dataset,Impact_on_Patient_Healthcare,is_using_Questionnaire,Impact_of_Synergy_Between_AI_ML_And_Robotic_Systems,is_using_Large_Language_Model,is_using_Multimodal_Model,is_using_Shared_Dataset,Is_Used_Shared_Dataset,C centralized_or_decentralized,data_collection_Method,is_USING_Survey_Data,is_USING_Questionnaire,is_USING_Large_Language_Model,is_USING_Multimodal_Model,is_USING_Shared_Dataset,Patinet_Level_Data,Iso_Patient_Level_Data,C centralized_or_D decentralized,clinical_data_type,Is_Used_Multimodal_Model,Cumulative_sum_analysis_used,Type_of_Patient_Level_Dataset,Airtightness_evaluation_of_Canadian_dwellings_and_influencing_factors_based_on_measured_data_and_predictive_models,AFFILIATION_COUNTRIES,CENTRALIZED_OR_DECentralized,IS_REAL_WORLD_DATA_SET,SIZE_OF_DATA_SET,AI_OR_ML_TOOLS_IN_THE_PAPER,GOVERNANCE_MODE,PATIENT_LEVEL_DATA,TYPE_OF_PATIENT_LEVEL_DATA,CLINICAL_DATA_TYPE,IS_USING_SURVEY_DATA,IS_USING_QUESTIONNAIRE,FEDERATED_LEARNING_TECHNIQUES,PRIVACY_PRESERVING_TECHNIQUES,DATA_COLLECTION_METHOD,IS_USING_LARGE_LANGUAGE_MODEL,IS_USING_MULTIMODAL_MODEL,IS_USING_SHARED_DATASET,MODALITIES,Is_Using MULTIMODAL_MODEL,Isof_Data_Set,Is_Using_Clinical_Data,Mental_Health_Dataset,Centralized_or_Distributed,Implicit_Dataset_Description,Affiliations,Iso_ML_Tools_In_The_Paper,Machine_Learning_Methods,Isof_Using_Survey_Data,Isof_Using_Questionnaire

In [ ]:
# len(canada_df)
canada_df.to_json('countries_affiliation_df_canada.json', orient='records')


In [ ]:
! pip install selenium


  Using cached trio_websocket-0.11.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached wsproto-1.2.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 16.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.0/476.0 kB 8.4 MB/s eta 0:00:00:00:01
Using cached trio_websocket-0.11.1-py3-none-any.whl (17 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 3.4 MB/s eta 0:00:00
Using cached wsproto-1.2.0-py3-none-any.whl (24 kB)
Using cached outcome-1.3.0.post0-py2.py3-none-any.whl (10 kB)
Using cached h11-0.14.0-py3-none-any.whl (58 kB)
  Attempting uninstall: attrs
    Found existing installation: attrs 23.1.0
    Uninstalling attrs-23.1.0:
      Successfully uninstalled attrs-23.1.0


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

service = Service('/opt/homebrew/bin/chromedriver')
options = Options()
options.headless = True  # Run Chrome in headless mode
driver = webdriver.Chrome(service=service, options=options)

driver.get('https://www.google.com')
print(driver.title)
driver.quit()


Google


In [ ]:
canada_df['PMID'].to_csv('canada_pmid.csv', index=False, header=False)


In [ ]:
import os
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
import pandas as pd

# Load the PMIDs from the CSV file
df_data = pd.read_csv('canada_pmid.csv', header=None, names=['PMID'])

# Set up Chrome driver for macOS M1
# Adjust this path to where you placed the chromedriver
chrome_driver_path = "/opt/homebrew/bin/chromedriver"
# Replace with your desired download folder path
download_folder = "/Users/yumcoder/Desktop/pubmed/download"

service = Service(chrome_driver_path)
options = webdriver.ChromeOptions()

# Set the download folder for Chrome
prefs = {"download.default_directory": download_folder}
options.add_experimental_option("prefs", prefs)

# Initialize the WebDriver
driver = webdriver.Chrome(service=service, options=options)

try:
    for index, row in df_data.iterrows():
        pmid = row['PMID']
        pdf_filename = os.path.join(download_folder, f"{pmid}.pdf")
        if index > 10:
            break

        # Check if the file already exists
        if os.path.exists(pdf_filename):
            print(f"File for PMID {pmid} already exists, skipping download.")
            continue

        # Open the PubMed article page
        driver.get(f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/")

        # Wait for the page to load and the full-text links to be visible
        wait = WebDriverWait(driver, 10)

        # Wait for the full-text links container to be present
        wait.until(EC.presence_of_element_located(
            (By.CLASS_NAME, "full-text-links-list")))

        # Find all full-text links
        try:
            links = wait.until(EC.presence_of_all_elements_located(
                (By.XPATH, "//div[@class='full-text-links-list']/a")))

            # Debug: Print information about each link using textContent
            for link in links:
                link_text = link.get_attribute('textContent').strip()
                link_href = link.get_attribute('href')
                print(f"Link text: {link_text}")
                print(f"Link href: {link_href}")

            # Optionally click on a specific link (e.g., PMC link)
            for link in links:
                link_text = link.get_attribute('textContent').strip()
                if "Free PMC article" in link_text:
                    link.click()
                    break

        except NoSuchElementException:
            print(f"Full-text links not found for PMID {pmid}.")
            continue

        # Check if a new tab opened
        window_handles = driver.window_handles
        if len(window_handles) > 1:
            # Switch to the new tab
            driver.switch_to.window(window_handles[-1])
        else:
            print("No new tab detected, continuing in the current tab.")

        # Wait for the new page or current page to load the PDF link
        wait.until(EC.presence_of_element_located(
            (By.XPATH, "//a[contains(@href, '.pdf')]")))

        # Find the download PDF link (if available)
        try:
            pdf_link = driver.find_element(
                By.XPATH, "//a[contains(@href, '.pdf')]")

            # Debug: Print the PDF link
            print(f"Found PDF link: {pdf_link}")
            print(f"PDF URL: {pdf_link.get_attribute('href')}")

            # Get the URL of the PDF file
            pdf_url = pdf_link.get_attribute("href")

            # Mimic a real browser request by adding headers
            headers = {
                "User-Agent": driver.execute_script("return navigator.userAgent;"),
                "Referer": driver.current_url,
                "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
            }

            # Download the PDF file using requests with headers
            response = requests.get(pdf_url, headers=headers, stream=True)

            # Ensure the request was successful
            if response.status_code == 200:
                with open(pdf_filename, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=1024):
                        if chunk:  # filter out keep-alive new chunks
                            f.write(chunk)
                print(f"Downloaded full text PDF to: {pdf_filename}")
            else:
                print(
                    f"Failed to download PDF for PMID {pmid}. Status code: {response.status_code}")

        except NoSuchElementException:
            print(f"PDF link not found for PMID {pmid}.")

finally:
    # Close the browser
    driver.quit()


File for PMID 28350526 already exists, skipping download.
File for PMID 28444633 already exists, skipping download.
File for PMID 28550657 already exists, skipping download.
File for PMID 28589480 already exists, skipping download.
File for PMID 28596271 already exists, skipping download.
Link text: Elsevier Science
Link href: https://linkinghub.elsevier.com/retrieve/pii/S0920-9964(17)30302-X
Link text: Elsevier Science
Link href: https://linkinghub.elsevier.com/retrieve/pii/S0920-9964(17)30302-X
No new tab detected, continuing in the current tab.


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=127.0.6533.101)
Stacktrace:
0   chromedriver                        0x0000000100a090b8 cxxbridge1$str$ptr + 1887276
1   chromedriver                        0x0000000100a01794 cxxbridge1$str$ptr + 1856264
2   chromedriver                        0x000000010061082c cxxbridge1$string$len + 88524
3   chromedriver                        0x00000001005ebe50 core::str::slice_error_fail::he7b2aa4898bc357e + 3908
4   chromedriver                        0x0000000100679574 cxxbridge1$string$len + 517908
5   chromedriver                        0x000000010068bdd8 cxxbridge1$string$len + 593784
6   chromedriver                        0x0000000100649474 cxxbridge1$string$len + 321044
7   chromedriver                        0x000000010064a0e4 cxxbridge1$string$len + 324228
8   chromedriver                        0x00000001009d0a9c cxxbridge1$str$ptr + 1656336
9   ch

In [ ]:
import os
import ecdsa
import hashlib
import base58


def private_key_to_address(private_key_bytes):
    sk = ecdsa.SigningKey.from_string(private_key_bytes, curve=ecdsa.SECP256k1)
    vk = sk.verifying_key
    public_key_bytes = b'\x04' + vk.to_string()
    sha256_pk = hashlib.sha256(public_key_bytes).digest()
    ripemd160 = hashlib.new('ripemd160')
    ripemd160.update(sha256_pk)
    hashed_pk = ripemd160.digest()
    prefix = b'\x00' + hashed_pk
    checksum = hashlib.sha256(hashlib.sha256(prefix).digest()).digest()[:4]
    return base58.b58encode(prefix + checksum).decode()


# Brute-force loop (useless, but technically correct)
target = "1PMycacnJaSqwwJqjawXBErnLsZ7RkXUAs"
while True:
    pk = os.urandom(32)
    addr = private_key_to_address(pk)
    print("tes:", pk.hex())

    if addr == target:
        print("Match found!", pk.hex())
        break
